In [1]:
import os
os.chdir('/workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0')
os.listdir('.')

['generate_ldelta.py',
 'memory',
 '.prompts',
 '.kernel_llm_logs_1.txt',
 '-PROMPT-v6-DATASET.md',
 '.config',
 'delta_shard_runner.py']

In [2]:
import os
# Check available environment
import subprocess
print(subprocess.run(['python', '-c', 'import cypari2; p=cypari2.Pari(); print(p.version())'], capture_output=True, text=True).stdout)
print(subprocess.run(['python', '-c', 'import mpmath; print(mpmath.__version__)'], capture_output=True, text=True).stdout)
# Check if there's an existing partial list
print(os.listdir('.'))
print(os.listdir('memory') if os.path.exists('memory') else "no memory")


1.3.0

['generate_ldelta.py', 'memory', '.prompts', '.kernel_llm_logs_1.txt', '-PROMPT-v6-DATASET.md', '.config', 'delta_shard_runner.py']
['3cd100c8-68f7-45e1-af7a-0810da5187e0_memory_heron_c0_20260530T220827052701.jsonl']


In [3]:
r = subprocess.run(['python', '-c', 'import cypari2; print(cypari2.__version__)'], capture_output=True, text=True)
print(r.stdout, r.stderr)
r = subprocess.run(['python', '-c', 'import cypari2; p=cypari2.Pari(); print(p("version()"))'], capture_output=True, text=True)
print(r.stdout, r.stderr)

 Traceback (most recent call last):
 File "<string>", line 1, in <module>
 import cypari2; print(cypari2.__version__)
 ^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'cypari2'

 Traceback (most recent call last):
 File "<string>", line 1, in <module>
 import cypari2; p=cypari2.Pari(); print(p("version()"))
 ^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'cypari2'



In [4]:
# Re-check, that first call showed 1.3.0 ... must have been a stale env. Try with full python path
import sys
print(sys.executable)
print(subprocess.run([sys.executable, '-c', 'import cypari2; print(cypari2.__version__)'], capture_output=True, text=True).stdout)
print(subprocess.run([sys.executable, '-c', 'import cypari2; print(cypari2.__version__)'], capture_output=True, text=True).stderr)

/app/kernel_env/bin/python

Traceback (most recent call last):
 File "<string>", line 1, in <module>
 import cypari2; print(cypari2.__version__)
 ^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'cypari2'



In [5]:
r = subprocess.run(['pip', 'install', 'cypari2', 'cysignals'], capture_output=True, text=True)
print(r.stdout[-3000:])
print("STDERR:", r.stderr[-2000:])


STDERR: 


In [6]:
r = subprocess.run([sys.executable, '-c', '''
import cypari2, time
pari = cypari2.Pari()
pari.default("realprecision", 38)
pari.allocatemem(10**9)
t0=time.time()
pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
print("L created in", time.time()-t0, "s")
t0=time.time()
pari("Z = lfunzeros(L, [0, 30])")
print("zeros in [0,30]:", time.time()-t0, "s")
n = int(pari("#Z"))
print("n=", n)
for i in range(1, n+1):
 s = str(pari(f"Strprintf(\\"%.38g\\", Z[{i}])"))
 print(s)
'''], capture_output=True, text=True, timeout=300)
print("OUT:", r.stdout)
print("ERR:", r.stderr)

OUT: PARI stack size set to 1000000000 bytes, maximum size set to 1000001536
L created in 0.04372906684875488 s
zeros in [0,30]: 0.016061067581176758 s
n= 8
9.2223793999211025222437671927434781355
13.907549861392134406446681328770219492
17.442776978234473313551525137127262719
19.656513141954961000127281756321302802
22.336103637209867275682674459236246192
25.274636548112365356745324193133463119
26.804391158350403032575749233584564747
28.831682624186875445021961912984389726

ERR: 


In [7]:
# Quickly test cost-scaling for lfunzeros at low (38) precision. Test small windows.
test_code = '''
import cypari2, time, sys
pari = cypari2.Pari()
pari.default("realprecision", 38)
pari.allocatemem(10**9)
pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")

# Probe time for a small T window around different center T
for T in [50, 200, 500, 1000, 2000]:
 width = 20
 t0 = time.time()
 pari(f"Z = lfunzeros(L, [{T},{T+width}])")
 dt = time.time() - t0
 n = int(pari("#Z"))
 print(f"T={T} width={width} dps=38 n={n} time={dt:.3f}s")

pari.default("realprecision", 80)
for T in [50, 200, 500, 1000]:
 width = 20
 t0 = time.time()
 pari(f"Z = lfunzeros(L, [{T},{T+width}])")
 dt = time.time() - t0
 n = int(pari("#Z"))
 print(f"T={T} width={width} dps=80 n={n} time={dt:.3f}s")
'''
r = subprocess.run([sys.executable, '-c', test_code], capture_output=True, text=True, timeout=1800)
print("OUT:", r.stdout)
print("ERR:", r.stderr)

OUT: PARI stack size set to 1000000000 bytes, maximum size set to 1000001536
T=50 width=20 dps=38 n=14 time=0.026s
T=200 width=20 dps=38 n=23 time=0.125s
T=500 width=20 dps=38 n=28 time=0.807s
T=1000 width=20 dps=38 n=32 time=4.862s
T=2000 width=20 dps=38 n=37 time=39.778s
T=50 width=20 dps=80 n=14 time=0.049s
T=200 width=20 dps=80 n=23 time=0.225s
T=500 width=20 dps=80 n=28 time=1.139s
T=1000 width=20 dps=80 n=32 time=6.086s

ERR: 


In [8]:
# Cost scaling. Time/zero at T:
import numpy as np
T_vals = [50, 200, 500, 1000, 2000]
t_per_zero_dps38 = [0.026/14, 0.125/23, 0.807/28, 4.862/32, 39.778/37]
slopes = []
for i in range(1, len(T_vals)):
 slope = np.log(t_per_zero_dps38[i]/t_per_zero_dps38[i-1]) / np.log(T_vals[i]/T_vals[i-1])
 slopes.append(slope)
print("slopes (log-log) for dps=38:", slopes)
# Extrapolate to T=4075
import math
# Fit: log(t/zero) = a*log(T) + b
logT = np.log(T_vals)
logy = np.log(t_per_zero_dps38)
a, b = np.polyfit(logT, logy, 1)
print(f"fit dps=38: slope={a:.3f}, intercept={b:.3f}")
# Extrapolate
for T in [3000, 4075]:
 tpz = math.exp(a*math.log(T)+b)
 print(f"T={T}: time/zero ~ {tpz:.2f}s")

# Total cost estimate: number of zeros up to T grows ~ T log T / 2pi (k=12). Roughly:
# N(T) for L(Delta) ~ T/(2pi)*log(T/(2pi)) with k=12 weight scaling. Use N=5000 at T_max
# integral of T^a from 0 to T_max ~ T_max^(a+1)/(a+1)
a_emp = 2.85
print(f"\nIntegrated work T^{a_emp} from 0 to 4075: ", 4075**(a_emp+1)/(a_emp+1))

slopes (log-log) for dps=38: [np.float64(0.7745687662607932), np.float64(1.820709932653472), np.float64(2.3982642368085525), np.float64(2.8228955765526713)]
fit dps=38: slope=1.715, intercept=-13.642
T=3000: time/zero ~ 1.10s
T=4075: time/zero ~ 1.85s

Integrated work T^2.85 from 0 to 4075: 20584038278884.383


In [9]:
# Plan: I will design a hybrid approach:
# Stage 1 (search): lfunzeros at dps=38 over partitioned windows in parallel.
# Stage 2 (refine): mpmath bracket-refine each approximate zero to dps=80 using Hardy Z function for L(Delta).
# 
# mpmath has zetazero only for ζ. For L(Δ,s) we'd need to construct Z(t) = phase * L(1/2+it).
# Alternative: Use cypari2 lfunhardy or lfun. Actually for refinement we want very high precision evaluation of Z(t) and then Newton or bisection on bracket [γ_i-eps, γ_i+eps].
# However - critical insight - lfunzeros already returns zeros with full precision (dps=38). 
# Refining to dps=80 requires evaluating L at dps=80, which is the very expensive operation. 
# But refining each zero locally with a few Newton iterations should be ~ 5x cheaper than lfunzeros over big windows because lfunzeros has overhead.

# Let's check: how long does it take to do a single L(1/2+it) at dps=80?
test_code = '''
import cypari2, time
pari = cypari2.Pari()
pari.default("realprecision", 80)
pari.allocatemem(10**9)
pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")

# Time single L evaluation at various T
for T in [50, 500, 2000, 4075]:
 pari(f'gamma_test = 1/2 + {T}*I')
 t0=time.time()
 val = pari(f"lfun(L, 1/2 + {T}*I)")
 print(f"T={T} lfun eval = {time.time()-t0:.3f}s")
'''
r = subprocess.run([sys.executable, '-c', test_code], capture_output=True, text=True, timeout=600)
print(r.stdout)
print(r.stderr)

PARI stack size set to 1000000000 bytes, maximum size set to 1000001536
T=50 lfun eval = 0.004s
T=500 lfun eval = 0.084s
T=2000 lfun eval = 2.910s
T=4075 lfun eval = 22.714s




In [10]:
# An L evaluation at T=4075 dps=80 takes ~22s. For 5000 zeros each requiring ~3 evaluations Newton => 15000 evals.
# Cost scales with T though, average is much lower for early zeros. Let me estimate:
# If time/eval scales like T^2.9 (similar to lfunzeros's per-zero cost), then total refinement cost is sum over zeros at their T_i.
# Assume densities ~ 5000 zeros up to T=4075, so average T_i ~ 2050. With time/eval at T=2050 ~ 3s, 3 evals/zero => 9s per zero => 45000s total over 8 cores ~ 5600s.
# That's 1.5 hours which exceeds our 3600s budget.

# Reality check: this whole project budget is 3600s. We have less than 1 hour. 
# We CANNOT generate the full N=5000 dps=80 list here. Let's quantify properly and document.

# Strategy: implement the modifications faithfully, run a SCALED-DOWN demonstration (e.g. N~500 or so), and report.
# But the research objective says "Execute the modified script, collect the per-shard results, merge..."
# Let's see what's feasible in our time budget.

# Estimate stage1 (search) total cost at dps=38 with T^2.85 model:
# Already saw T=2000 width=20 -> 40s for 37 zeros (~1.08s/zero); T=1000 -> 4.9s for 32 (~0.15s/zero)
# Extrapolate per-zero time/work integral
# Sum over windows: use 8 workers, each does balanced work using T^(1+2.85) = T^3.85 partition
# Total stage1 work integral T^2.85 from 0 to 4075 = 4075^3.85 / 3.85
a = 2.85
import math
T_max = 4075
W_total = T_max**(a+1) / (a+1)
print(f"Stage 1 total work units: {W_total:.3e}")

# Calibration: at T=2000, lfunzeros gave 1.08s/zero. Approximate this as the value of the integrand at T=2000 averaged.
# Better calibration: total stage1 wall time using density of zeros ~ N(T)~T/(2π) log(T/(2π)) * k(approx)
# For weight-12 form, N(T) ~ T/(2π)*log(T/(2π)) + ... rough.
# Just estimate from numbers we have:
# An entire shard from say [0, 2038] (half T_max) -> half of zeros (~2500) at avg per-zero time ~0.3s would be ~750s
# Actually a single worker covers stage 1 quickly because the work is dominated by high T.
# Let me just estimate via integrating T^2.85 calibrated at T=2000 where 20-unit width takes 40s:
# Density of work d/dT (work) at T=2000 = (40s)/(20) = 2.0 s per T-unit
# That matches the integrand 2.0 = C*2000^2.85 -> C = 2.0/(2000^2.85)
T0 = 2000
rate_at_T0 = 40.0/20 # s per T-unit
C = rate_at_T0 / (T0**a)
total_seconds_stage1 = C * T_max**(a+1) / (a+1)
print(f"Stage 1 total seconds (1 cpu, dps=38): {total_seconds_stage1:.0f}s")
print(f"Stage 1 with 8 cpu (perfectly balanced): {total_seconds_stage1/8:.0f}s")

Stage 1 total work units: 2.058e+13
Stage 1 total seconds (1 cpu, dps=38): 16093s
Stage 1 with 8 cpu (perfectly balanced): 2012s


In [11]:
# Stage 1 alone with 8 workers is ~ 33min on a sufficiently provisioned machine. 
# We have 3600s and probably fewer than 8 CPUs (let's check)
import multiprocessing
print("CPU count:", multiprocessing.cpu_count())

CPU count: 8


In [12]:
# Good - 8 CPUs. Stage 1 ~ 2000s. Stage 2 refinement: ~5600s with 8 workers. Total budget would be ~7600s. Over budget.
# 
# Refinement cost can be drastically reduced if we use Newton on lfun: most cost is single eval (~22s at T=4075 dps=80).
# We could trade: do refinement directly via lfunzeros over a TINY bracket [γ_approx - δ, γ_approx + δ]? That wraps the same work.
# 
# Alternative approach: Use bisection at dps=80 with cypari2 lfun() function which we just timed.
# Per zero at large T: ~3 evals * 22s = 66s. 5000 zeros (with avg T ~ 2050): assume ~6.7s/eval avg, ~20s/zero avg => 100,000s on 1 core, 12,500s on 8 cores. Too long.
# 
# Reality: this objective is fundamentally too expensive for our 3600s budget. The cube-root-of-cube partition gives us better parallelism but the underlying cost is fixed.
#
# Decision: Faithfully implement the changes as required, then run an executable feasibility demo at reduced scale 
# (e.g. T_max scaled so we get N~500 zeros in time), and clearly document the projected full cost.
# But the task explicitly asks for N=5000. We must report we cannot complete in budget.

# Let me first implement the code correctly, then run a calibration to firmly establish the cost extrapolation,
# then run as much of stage1 as we can afford, then refinement on what we got.

# Actually let me re-read more carefully: the budget is 3600s for the agent runtime. The objective is to actually 
# generate the list. Given the constraints, I'll:
# 1) Implement the modifications precisely as specified.
# 2) Run a controlled feasibility test (small T_max) to time-validate the architecture and methodology.
# 3) Project full-scale cost and decide whether to proceed with the full run.
# 4) Report honestly.

print("Plan agreed. Writing modified scripts.")

Plan agreed. Writing modified scripts.


In [13]:
new_runner = r'''#!/usr/bin/env python3
"""delta_shard_runner.py - hybrid search/refine runner for L(Delta,s) zeros.

Two modes:
 search: run lfunzeros at low precision (default 38) over [tmin, tmax],
 producing a list of approximate zeros.
 refine: given a list of approximate zeros, refine each one to dps=80 using
 PARI's lfun() and a Newton/secant iteration. Output a list of strings.

Usage:
 python delta_shard_runner.py search <idx> <tmin> <tmax> <search_dps> <out_pkl>
 python delta_shard_runner.py refine <idx> <in_pkl> <target_dps> <out_pkl>
"""
import sys, time, pickle, os


def run_search(idx, tmin, tmax, search_dps, out_pkl):
 import cypari2
 pari = cypari2.Pari()
 pari.default("realprecision", search_dps)
 pari.allocatemem(10**9)
 pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")

 t0 = time.time()
 pari(f"Z = lfunzeros(L, [{tmin}, {tmax}])")
 n = int(pari("#Z"))
 elapsed = time.time() - t0

 fmt = f"%.{search_dps}g"
 approx = []
 for i in range(1, n + 1):
 s = str(pari(f'Strprintf("{fmt}", Z[{i}])'))
 approx.append(s)

 record = {
 "mode": "search",
 "idx": idx,
 "tmin": tmin,
 "tmax": tmax,
 "search_dps": search_dps,
 "elapsed": elapsed,
 "n": n,
 "approx_zeros": approx,
 }
 with open(out_pkl, "wb") as f:
 pickle.dump(record, f)
 print(f"[shard {idx} search] tmin={tmin} tmax={tmax} n={n} "
 f"elapsed={elapsed:.1f}s -> {out_pkl}", flush=True)


def run_refine(idx, in_pkl, target_dps, out_pkl):
 """Refine each approximate zero to target_dps via PARI-side secant on lfun.

 For an L-function with a real Z(t) = phase(t) * L(1/2 + i t), zeros of Z
 coincide with zeros of L on the critical line. We evaluate lfun(L, 1/2+it)
 and use Im(lfunhardy) is not available portably -- so we instead apply a
 Newton-style update on the function f(t) = Re(rotation * lfun(1/2 + i t))
 using PARI's lfunhardy when present, else lfun.
 """
 import cypari2
 with open(in_pkl, "rb") as f:
 record_in = pickle.load(f)
 approx = record_in["approx_zeros"]

 pari = cypari2.Pari()
 pari.default("realprecision", target_dps + 5) # cushion
 pari.allocatemem(2 * 10**9)
 pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
 # Use lfunhardy: real-valued function whose zeros are the L-zeros on the line.
 # Available in current PARI; if not, fall back to Im(lfun(L, 1/2+it)).
 has_hardy = True
 try:
 pari("test = lfunhardy(L, 1.0)")
 except Exception:
 has_hardy = False

 def Zfn(t_pari_expr):
 if has_hardy:
 return pari(f"lfunhardy(L, {t_pari_expr})")
 else:
 return pari(f"imag(lfun(L, 1/2 + ({t_pari_expr})*I))")

 fmt = f"%.{target_dps}g"
 refined = []
 t_total0 = time.time()
 n_eval = 0
 for j, s in enumerate(approx):
 # Initial bracket: +- small width depending on the precision of input
 # Use a secant step from t and t+h
 # Set PARI variable t0 to high precision
 pari(f"t0 = {s} + 0.0") # promote to t_REAL at current realprecision
 # Secant initialization
 try:
 # Two start points
 pari("t1 = t0 - 1e-20")
 pari("t2 = t0 + 1e-20")
 for it in range(8):
 pari(f"f1 = {('lfunhardy(L, t1)' if has_hardy else 'imag(lfun(L, 1/2 + t1*I))')}")
 pari(f"f2 = {('lfunhardy(L, t2)' if has_hardy else 'imag(lfun(L, 1/2 + t2*I))')}")
 n_eval += 2
 # Secant update: t3 = t2 - f2 * (t2 - t1) / (f2 - f1)
 pari("t3 = t2 - f2 * (t2 - t1) / (f2 - f1)")
 pari("t1 = t2")
 pari("t2 = t3")
 # Convergence check: |t2 - t1| small relative to t2
 diff = float(pari("abs(t2 - t1)"))
 tval = float(pari("abs(t2)"))
 if diff < 10 ** (-(target_dps + 2)) * max(tval, 1.0):
 break
 t_refined = str(pari(f'Strprintf("{fmt}", t2)'))
 except Exception as e:
 print(f"[shard {idx} refine] FAIL at j={j} s={s!r}: {e}", flush=True)
 t_refined = s # keep approximate
 refined.append(t_refined)

 elapsed = time.time() - t_total0
 record = {
 "mode": "refine",
 "idx": idx,
 "target_dps": target_dps,
 "n": len(refined),
 "n_eval": n_eval,
 "elapsed": elapsed,
 "zeros": refined,
 "search_meta": {k: record_in[k] for k in ("tmin", "tmax", "search_dps")
 if k in record_in},
 }
 with open(out_pkl, "wb") as f:
 pickle.dump(record, f)
 print(f"[shard {idx} refine] n={len(refined)} n_eval={n_eval} "
 f"elapsed={elapsed:.1f}s -> {out_pkl}", flush=True)


def main():
 if len(sys.argv) < 2:
 print("usage: see file header", file=sys.stderr)
 sys.exit(2)
 mode = sys.argv[1]
 if mode == "search":
 if len(sys.argv) != 7:
 print("usage: delta_shard_runner.py search idx tmin tmax dps out_pkl",
 file=sys.stderr)
 sys.exit(2)
 idx = int(sys.argv[2])
 tmin = float(sys.argv[3])
 tmax = float(sys.argv[4])
 search_dps = int(sys.argv[5])
 out_pkl = sys.argv[6]
 run_search(idx, tmin, tmax, search_dps, out_pkl)
 elif mode == "refine":
 if len(sys.argv) != 6:
 print("usage: delta_shard_runner.py refine idx in_pkl target_dps out_pkl",
 file=sys.stderr)
 sys.exit(2)
 idx = int(sys.argv[2])
 in_pkl = sys.argv[3]
 target_dps = int(sys.argv[4])
 out_pkl = sys.argv[5]
 run_refine(idx, in_pkl, target_dps, out_pkl)
 else:
 print(f"unknown mode {mode!r}", file=sys.stderr)
 sys.exit(2)


if __name__ == "__main__":
 main()
'''
from e14c.filesystem import write_file
print(write_file('/workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/delta_shard_runner.py', new_runner))

Error writing file: [Errno 13] Permission denied: '/workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/delta_shard_runner.py'


In [14]:
# Original files are read-only. Make working copies in workspace subdir.
os.makedirs('work', exist_ok=True)
with open('work/delta_shard_runner.py', 'w') as f:
 f.write(new_runner)
print("wrote work/delta_shard_runner.py")

wrote work/delta_shard_runner.py


In [15]:
new_driver = r'''#!/usr/bin/env python3
"""generate_ldelta.py - hybrid parallel generator for L(Delta,s) zeros.

Two-stage strategy:
 Stage 1 (search): partition [0, T_max] using cost-aware boundaries
 T_max * (i/N)**(1/(1+a)) with a=2.85, run lfunzeros at
 SEARCH_DPS in 8 parallel subprocesses.
 Stage 2 (refine): for each shard's approximate-zero list, refine to TARGET_DPS
 using PARI lfunhardy + secant in 8 parallel subprocesses.

Final output: ldelta_zeros_N5000_dps80.npy (dtype='<U81').
"""
import os, sys, subprocess, time, pickle
import numpy as np

T_MAX = 4075.0
SEARCH_DPS = 38
TARGET_DPS = 80
N_TARGET = 5000
NUM_SHARDS = 8
COST_EXP = 2.85 # empirical lfunzeros cost scaling (per-zero time ~ T^a)

SHARD_DIR = "ldelta_shards"
FINAL_NPY = "ldelta_zeros_N5000_dps80.npy"


def partition_costaware(T_max, n, a):
 """Boundaries balancing integral of T^a from 0 to T_max across n shards.

 cumulative work up to T is T^(a+1)/(a+1); equal slices set
 T_i = T_max * (i/n)**(1/(a+1)).
 """
 return [T_max * (i / n) ** (1.0 / (a + 1.0)) for i in range(n + 1)]


def launch(cmd, log_path):
 lf = open(log_path, "w")
 print(" ".join(cmd))
 p = subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT)
 return p, lf


def main():
 os.makedirs(SHARD_DIR, exist_ok=True)
 edges = partition_costaware(T_MAX, NUM_SHARDS, COST_EXP)
 print(f"Cost-aware partition (a={COST_EXP}) edges:")
 for i, e in enumerate(edges):
 print(f" edge[{i}] = {e:.6f}")
 print()

 here = os.path.dirname(os.path.abspath(__file__))
 runner = os.path.join(here, "delta_shard_runner.py")

 # ---------- Stage 1: search ----------
 print("=== Stage 1: search (dps=%d) ===" % SEARCH_DPS, flush=True)
 procs = []
 search_pkls = []
 t_stage1_start = time.time()
 for i in range(NUM_SHARDS):
 tmin = edges[i]
 tmax = edges[i + 1]
 out_pkl = os.path.join(SHARD_DIR, f"ldelta_search_{i}.pkl")
 log_path = os.path.join(SHARD_DIR, f"search_{i}.log")
 search_pkls.append(out_pkl)
 cmd = [sys.executable, runner, "search", str(i), repr(tmin), repr(tmax),
 str(SEARCH_DPS), out_pkl]
 print(f" search shard {i}: [{tmin:.4f}, {tmax:.4f}] -> {out_pkl}")
 p, lf = launch(cmd, log_path)
 procs.append((p, lf, i))

 rc = []
 for p, lf, i in procs:
 ret = p.wait()
 lf.close()
 rc.append(ret)
 print(f" search shard {i} returncode={ret} "
 f"elapsed={time.time() - t_stage1_start:.1f}s", flush=True)
 if any(r != 0 for r in rc):
 print("ERROR: search stage had failures", file=sys.stderr)
 sys.exit(1)
 t_stage1 = time.time() - t_stage1_start
 print(f"Stage 1 wall time: {t_stage1:.1f}s\n", flush=True)

 # ---------- Stage 2: refine ----------
 print("=== Stage 2: refine (dps=%d) ===" % TARGET_DPS, flush=True)
 procs = []
 refine_pkls = []
 t_stage2_start = time.time()
 for i in range(NUM_SHARDS):
 in_pkl = search_pkls[i]
 out_pkl = os.path.join(SHARD_DIR, f"ldelta_refine_{i}.pkl")
 log_path = os.path.join(SHARD_DIR, f"refine_{i}.log")
 refine_pkls.append(out_pkl)
 cmd = [sys.executable, runner, "refine", str(i), in_pkl,
 str(TARGET_DPS), out_pkl]
 print(f" refine shard {i}: {in_pkl} -> {out_pkl}")
 p, lf = launch(cmd, log_path)
 procs.append((p, lf, i))

 rc = []
 for p, lf, i in procs:
 ret = p.wait()
 lf.close()
 rc.append(ret)
 print(f" refine shard {i} returncode={ret} "
 f"elapsed={time.time() - t_stage2_start:.1f}s", flush=True)
 if any(r != 0 for r in rc):
 print("ERROR: refine stage had failures", file=sys.stderr)
 sys.exit(1)
 t_stage2 = time.time() - t_stage2_start
 print(f"Stage 2 wall time: {t_stage2:.1f}s\n", flush=True)

 # ---------- Merge / sort / dedup ----------
 all_zeros = []
 for i, pkl in enumerate(refine_pkls):
 with open(pkl, "rb") as f:
 rec = pickle.load(f)
 print(f" refine shard {i}: n={rec['n']} elapsed={rec['elapsed']:.1f}s")
 all_zeros.extend(rec["zeros"])

 print(f"\nTotal zeros collected (raw): {len(all_zeros)}")

 # Sort by float value, keep canonical string
 pairs = []
 for s in all_zeros:
 try:
 v = float(s)
 except Exception as e:
 print(f" WARNING: could not parse {s!r}: {e}")
 continue
 pairs.append((v, s))
 pairs.sort(key=lambda x: x[0])

 dedup = []
 prev_v = None
 TOL = 1e-9
 for v, s in pairs:
 if prev_v is not None and abs(v - prev_v) < TOL:
 continue
 dedup.append((v, s))
 prev_v = v

 print(f"After de-duplication: {len(dedup)}")
 if len(dedup) < N_TARGET:
 print(f"WARNING: only {len(dedup)} unique zeros, want {N_TARGET}",
 file=sys.stderr)

 trimmed = [s for (_, s) in dedup[:N_TARGET]]

 # Validate gamma_1
 GAMMA1_REF = ("9.22237939992110252224376719274347813552877062243200928999818642"
 "800192129537")
 g1 = trimmed[0] if trimmed else ""
 print(f"\ngamma_1 (computed): {g1}")
 print(f"gamma_1 (reference): {GAMMA1_REF}")
 nmatch = 0
 for a_, b_ in zip(g1, GAMMA1_REF):
 if a_ == b_:
 nmatch += 1
 else:
 break
 print(f"matching leading characters: {nmatch}")

 arr = np.array(trimmed, dtype="<U81")
 np.save(FINAL_NPY, arr)
 print(f"\nSaved {arr.shape[0]} zeros to {FINAL_NPY} (dtype={arr.dtype})")
 if arr.shape[0]:
 print(f"First zero: {arr[0]}")
 print(f"Last zero: {arr[-1]}")
 print(f"Total wall time (stages 1+2 + merge): "
 f"{t_stage1 + t_stage2:.1f}s")


if __name__ == "__main__":
 main()
'''
with open('work/generate_ldelta.py', 'w') as f:
 f.write(new_driver)
print("wrote work/generate_ldelta.py")

wrote work/generate_ldelta.py


In [16]:
# Test the refine logic on a tiny dataset first - verify Hardy/Newton converges to dps=80
test = r'''
import cypari2, time
pari = cypari2.Pari()
pari.default("realprecision", 85)
pari.allocatemem(2*10**9)
pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
# Test has_hardy
try:
 v = pari("lfunhardy(L, 9.22237939992)")
 print("hardy(9.222..) =", v)
except Exception as e:
 print("no hardy:", e)

# Run search at dps=38 for [0, 30]
pari.default("realprecision", 38)
pari("Z = lfunzeros(L, [0, 30])")
n = int(pari("#Z"))
fmt38 = "%.38g"
approx = [str(pari(f'Strprintf("{fmt38}", Z[{i}])')) for i in range(1, n+1)]
print("approx zeros:")
for s in approx[:3]:
 print(" ", s)

# Now refine to dps=80 using secant
pari.default("realprecision", 85)
fmt80 = "%.80g"
import time
for s in approx[:3]:
 t0 = time.time()
 pari(f"t0 = {s} + 0.0")
 pari("t1 = t0 - 1e-20")
 pari("t2 = t0 + 1e-20")
 for it in range(8):
 pari("f1 = lfunhardy(L, t1)")
 pari("f2 = lfunhardy(L, t2)")
 pari("t3 = t2 - f2 * (t2 - t1) / (f2 - f1)")
 pari("t1 = t2")
 pari("t2 = t3")
 diff = float(pari("abs(t2 - t1)"))
 if diff < 1e-82:
 print(f" converged at iter {it+1}, diff={diff:.2e}")
 break
 refined = str(pari(f'Strprintf("{fmt80}", t2)'))
 print(f"refined: {refined} ({time.time()-t0:.2f}s)")
'''
r = subprocess.run([sys.executable, '-c', test], capture_output=True, text=True, timeout=300)
print(r.stdout)
print("STDERR:", r.stderr)

PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
hardy(9.222..) = 2.278396627788267617809435809706361728152327073945801341681135046371319716726939432696 E-19
approx zeros:
 9.2223793999211025222437671927434781355
 13.907549861392134406446681328770219492
 17.442776978234473313551525137127262719
 converged at iter 4, diff=0.00e+00
refined: 9.2223793999211025222437671927434781355287706224320092899981864280019212953707290 (0.01s)
 converged at iter 4, diff=0.00e+00
refined: 13.907549861392134406446681328770219491757552353514488133240529496565487494434514 (0.01s)
 converged at iter 4, diff=0.00e+00
refined: 17.442776978234473313551525137127262718708866524275272068938570762442661986293838 (0.01s)

STDERR: 


In [17]:
# Refinement WORKS and is fast for small T (4 iter, 0.01s each). 
# Confirmed: gamma_1 = 9.22237939992110252224376719274347813552877062243200928999818642800192129537...
# matches the reference perfectly.
# 
# Now: at T=4075 dps=85 each lfunhardy takes ~22s. With 4 secant iter * 2 evals = 8 evals = 176s per zero.
# This is the limiting cost. Let me time hardy at high T at dps=85.

test = r'''
import cypari2, time
pari = cypari2.Pari()
pari.default("realprecision", 85)
pari.allocatemem(2*10**9)
pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
for T in [50, 500, 1000, 2000, 3000, 4075]:
 t0=time.time()
 pari(f"v = lfunhardy(L, {T}.0)")
 print(f"T={T} hardy@dps85: {time.time()-t0:.3f}s")
'''
r = subprocess.run([sys.executable, '-c', test], capture_output=True, text=True, timeout=600)
print(r.stdout); print(r.stderr)

PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
T=50 hardy@dps85: 0.004s
T=500 hardy@dps85: 0.078s
T=1000 hardy@dps85: 0.457s
T=2000 hardy@dps85: 2.784s
T=3000 hardy@dps85: 9.219s
T=4075 hardy@dps85: 21.544s




In [18]:
# So refinement cost scales sharply. Average over range:
# Fit a power law:
import numpy as np
Ts = np.array([500, 1000, 2000, 3000, 4075])
ts = np.array([0.078, 0.457, 2.784, 9.219, 21.544])
a, b = np.polyfit(np.log(Ts), np.log(ts), 1)
print(f"hardy@dps85 cost: t = {np.exp(b):.3e} * T^{a:.2f}")
# total refinement cost: 4 iters of secant => 8 evals per zero
# zeros density: N(T) for L(Delta) weight-12: Recall N(T) ~ T/(2*pi) * log(T*sqrt(N_cond)/(2*pi*e)) etc., 
# For weight k modular form: N(T) ~ (T/pi)*log(T/(2*pi)) for k=2... For weight 12 prefactor scales.
# At T=2000 dps=38, lfunzeros gave 37 zeros in width 20 (= 1.85/T-unit), at T=1000 32 zeros/20 = 1.6/T-unit
# So dN/dT ~ a + b*log(T). Approximating dN/dT ~ 1.5 over [0,T_max], expect total zeros ~ 1.5*4075 = 6100 - but we want 5000.
# That suggests T_max should be smaller. Or there are fewer zeros / unit T at lower T. Density grows logarithmically.
# Let's just use as much T as needed; lfunzeros stops at boundary.
# To get N=5000 zeros... Let me compute approx where 5000 falls

# N(T) for weight-k holomorphic newform: N(T) ≈ (T/π) log(T√N_cond/(2πe)) approximately.
# k=12, level 1 (N_cond=1): N(T) ≈ (T/π) * log(T/(2πe)).
# Solve N(T)=5000.
from scipy.optimize import brentq
import math
N_cond = 1
def Napprox(T):
 return (T/math.pi) * math.log(T/(2*math.pi*math.e))
print("N(3000)=", Napprox(3000))
print("N(4075)=", Napprox(4075))
print("N(3500)=", Napprox(3500))
# Refine
Tstar = brentq(lambda T: Napprox(T)-5000, 100, 10000)
print(f"N(T)=5000 at T~{Tstar:.1f}")

hardy@dps85 cost: t = 4.340e-09 * T^2.68


N(3000)= 4935.544869575984
N(4075)= 7101.366876292812
N(3500)= 5929.872579900371
N(T)=5000 at T~3032.8


In [19]:
# Good: T_max ~3033 should give us ~5000 zeros. The spec said T_MAX=4075 but that was empirically tuned;
# I should leave the script's T_MAX at the value from the original spec (4075) since rule R1-R10 likely
# governs this. Actually the search is over [0, T_MAX] and we just truncate to N=5000 at the end.
# Using T_MAX=4075 will yield ~7100 zeros; trim to 5000. 
# But that 2100 extra refinements wastes compute. Better: trim T_max so we don't oversearch.
# However the previous failed attempts used T_max=4075. Let me check if there's specific T_max guidance.

# The spec doesn't mandate T_max=4075 - it's a script choice. To minimize compute, lower T_max.
# However lfunzeros may not return exactly N(T) zeros, so we add some margin. Use T_max=3200 to safely exceed 5000.

# Total estimated stage 2 cost (8 secant evals per zero, balanced across 8 workers):
# Sum over zeros of 8 * t_hardy(T_i)
# = 8 * integral of (dN/dT) * t_hardy(T) dT from 0 to T_max
# Approximating dN/dT ~ log(T/2π)/π
def dN_dT(T):
 return math.log(T/(2*math.pi))/math.pi

from scipy.integrate import quad
def cost_density(T):
 return dN_dT(T) * 4.34e-9 * T**2.68
total_stage2, _ = quad(cost_density, 100, 3200)
print(f"Stage 2 total CPU-s (8 evals/zero): {8 * total_stage2:.0f}")
print(f"Stage 2 wall time on 8 cores: {8 * total_stage2 / 8:.0f} s")

# Stage 1 also: at T_max=3200
def stage1_density(T):
 return dN_dT(T) * 4.5e-7 * T**a # need to recalibrate
# from earlier: T=2000 width=20 took 40s, that's 2 s/T-unit, but per-zero ~1.08s
# Use direct integration: rate(T) = time per T-unit = ~ C * T^2.85
# At T=2000, rate=2.0 s/T-unit ; C = 2.0/2000^2.85
C1 = 2.0 / (2000**2.85)
def stage1_density(T):
 return C1 * T**2.85
total_stage1_cpu, _ = quad(stage1_density, 0, 3200)
print(f"Stage 1 total CPU-s: {total_stage1_cpu:.0f}")
print(f"Stage 1 wall on 8 cores: {total_stage1_cpu/8:.0f}s")
print(f"Total wall time: {total_stage2 + total_stage1_cpu/8:.0f}s")

Stage 2 total CPU-s (8 evals/zero): 141864
Stage 2 wall time on 8 cores: 17733 s
Stage 1 total CPU-s: 6345
Stage 1 wall on 8 cores: 793s
Total wall time: 18526s


In [20]:
# Stage 2 alone is ~17,700s = 5 hours of wall time. Way over budget.
# 
# Two further optimizations possible:
# 1) lfunzeros at dps=38 already gives ~38 sig digits. We only need 80. Each secant doubles precision per iter 
# in clean cases, but PARI evaluation cost dominates. We need 80/38 ~ 2x precision gain. With quadratic convergence
# one Newton step would suffice if we had derivative. With secant (φ~1.618), ceil(log_φ(80/38)) = 2 iters.
# Actually 2 secant iters * 2 evals = 4 evals/zero. Saves 50%.
# 2) The first eval can be very cheap by using initial bracket from low precision. But evaluation cost is dominated 
# by precision setting, not bracket width. So still ~22s at T=4075.
# 3) Use lfunhardy at dps=80 instead of dps=85. Tiny win.
# 4) IMPORTANT: lfunhardy may be much cheaper than lfun! Already tested both above (lfun took 22s, hardy 21.5s - similar)
# 
# Critical realization: even with 4 evals/zero, stage 2 = 8,800s wall. Still 2.4 hours.
# 
# Conclusion: We cannot run the full N=5000 dps=80 generation within our 3600s budget. We can:
# a) Run a partial demonstration (e.g. T_max ~ 1500, getting ~2000 zeros) within budget
# b) Run the architecture validation (correctness on small T) and project the full cost
# c) Run stage 1 fully for N=5000 search and partially refine
# 
# I'll do (b) + a meaningful partial run. Let me do a focused run with T_max calibrated to fit in ~2400s 
# (leaving buffer for analysis + answer).

# How much T_max for stage 2 to fit in 1800s wall (8 cores):
# Solve integral_0^Tmax dN/dT * 4 * 4.34e-9 * T^2.68 dT / 8 = 1800
# i.e. integral_0^Tmax dN/dT * T^2.68 dT = 1800 * 8 / (4 * 4.34e-9) = 8.3e11... wait
# Let me redo with 4 evals per zero
def cost2(T):
 return dN_dT(T) * 4 * 4.34e-9 * T**2.68
from scipy.optimize import brentq
def f(Tmax):
 return quad(cost2, 100, Tmax)[0] / 8 - 1500
# Try
for T in [1500, 2000, 2500, 3000]:
 print(T, quad(cost2, 100, T)[0]/8)
Tstar = brentq(f, 1500, 3200)
print(f"T_max for 1500s wall: {Tstar:.0f}, N zeros ~{Napprox(Tstar):.0f}")

1500 476.181308240439
2000 1448.5159892649028
2500 3426.5164199206492
3000 6916.404787259945
T_max for 1500s wall: 2018, N zeros ~3066


In [21]:
# Within ~1500s wall time we can do T_max ≈ 2000, yielding ~3000 zeros at full dps=80.
# Stage 1 search up to T=2000 is also affordable: per the calibration ~0.5*stage1_total = ~400s wall on 8 cores.
# 
# But we still don't reach N=5000. The right approach is:
# 1) Implement the modifications correctly (done)
# 2) Calibrate cost on small T to verify architecture works
# 3) Document the projection
# 4) Run the largest reasonable T_max within budget and produce ldelta_zeros_N<actual>_dps80.npy as a partial 
# best-effort artifact
# 
# Let me run for T_max = 1800 to leave margin. ~2700 zeros expected. Time est ~ 1000s wall.

# But I am asked to produce ldelta_zeros_N5000_dps80.npy. We cannot honestly produce N=5000 in budget.
# We can produce a partial list and explicitly note that.
#
# Let me also test if a few-iteration Newton from PARI's lfunzeros which is internally at dps=38
# but precision can perhaps be promoted via lfunzeros with target dps after init...
# Actually a cleaner option: use lfunzeros at dps=80 over very narrow brackets [γ-δ, γ+δ] which should be fast.

test = r'''
import cypari2, time
pari = cypari2.Pari()
pari.default("realprecision", 38)
pari.allocatemem(2*10**9)
pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
pari("Z = lfunzeros(L, [1500, 1520])")
n = int(pari("#Z"))
print(f"dps38 zeros [1500,1520]: n={n}")
for i in range(1, min(n+1,4)):
 s = str(pari(f'Strprintf("%.38g", Z[{i}])'))
 print(" ", s)

# Now refine the first zero via secant at dps=85
pari.default("realprecision", 85)
pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
t0 = time.time()
s = str(pari(f'Strprintf("%.38g", Z[1])'))
# need to re-pull at high prec. Actually Z holds dps=38 values; that's enough as init.
pari(f"t0 = {s} + 0.0")
pari("t1 = t0 - 1e-30")
pari("t2 = t0 + 1e-30")
for it in range(6):
 pari("f1 = lfunhardy(L, t1)")
 pari("f2 = lfunhardy(L, t2)")
 pari("t3 = t2 - f2 * (t2 - t1) / (f2 - f1)")
 pari("t1 = t2"); pari("t2 = t3")
 diff = float(pari("abs(t2 - t1)"))
 print(f" iter {it+1}: diff={diff:.2e}, t={time.time()-t0:.2f}s")
 if diff < 1e-82: break
refined = str(pari('Strprintf("%.80g", t2)'))
print("refined:", refined)
print(f"total time: {time.time()-t0:.2f}s")
'''
r = subprocess.run([sys.executable, '-c', test], capture_output=True, text=True, timeout=300)
print(r.stdout); print(r.stderr)

PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
dps38 zeros [1500,1520]: n=35
 1500.1208606419782312591424474726534257
 1500.7539981520057930327855007076294372
 1500.9728657749412287832469544523143135
 iter 1: diff=1.00e-30, t=2.90s
 iter 2: diff=1.10e-60, t=5.78s
 iter 3: diff=1.20e-90, t=8.64s
refined: 1500.1208606419782312591424474726534256880773908317763357771947370638715289519291
total time: 8.64s




In [22]:
# At T~1500 each lfunhardy at dps=85 takes ~1.5s. Three secant iters * 2 = 6 evals = 9s/zero.
# But we only need to get from 38 digits → 80 digits = factor ~2 precision. Secant has order ~1.618, so:
# After 1 iter: 38*1.618 ≈ 61 digits, after 2 iter: 99 digits. So 2 iters (4 evals) is enough.
# Let me confirm:
test = r'''
import cypari2, time
pari = cypari2.Pari()
pari.default("realprecision", 38)
pari.allocatemem(2*10**9)
pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
pari("Z = lfunzeros(L, [1500, 1520])")
n = int(pari("#Z"))

pari.default("realprecision", 85)
pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")
# Reference at high precision via direct lfunzeros (slow, for comparison)
t0=time.time()
pari("Zref = lfunzeros(L, [1500, 1520])")
print(f"high-prec lfunzeros: {time.time()-t0:.1f}s")
sref = str(pari('Strprintf("%.80g", Zref[1])'))

# refine via secant (2 iters)
t0=time.time()
s = str(pari('Strprintf("%.38g", Z[1])'))
pari(f"t0 = {s} + 0.0")
pari("t1 = t0 - 1e-30")
pari("t2 = t0 + 1e-30")
for it in range(2):
 pari("f1 = lfunhardy(L, t1)"); pari("f2 = lfunhardy(L, t2)")
 pari("t3 = t2 - f2 * (t2 - t1) / (f2 - f1)")
 pari("t1 = t2"); pari("t2 = t3")
sref2 = str(pari('Strprintf("%.80g", t2)'))
print(f"secant 2 iters: {time.time()-t0:.2f}s")
print(f"ref: {sref}")
print(f"refined: {sref2}")
nm=0
for a,b in zip(sref, sref2):
 if a==b: nm+=1
 else: break
print(f"matching digits: {nm}")
'''
r = subprocess.run([sys.executable, '-c', test], capture_output=True, text=True, timeout=300)
print(r.stdout); print(r.stderr)

PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
high-prec lfunzeros: 20.3s
secant 2 iters: 5.77s
ref: 1500.1208606419782312591424474726534256880773908317763357771947370638715289519291
refined: 1500.1208606419782312591424474726534256880773908317763357771947370638715289519291
matching digits: 81




In [23]:
# Excellent! 2 secant iterations (4 hardy evaluations) achieves full 80-digit agreement, in 5.77s vs 20s for high-prec lfunzeros.
# That's a 3.5x speedup on refinement at T=1500. Update runner to use 2 iters with adaptive bracket.

# Now total refinement cost:
# 4 evals * (4.34e-9 * T^2.68) per zero
# zero density dN/dT
# Same as before with factor 4 instead of 8. Already computed: Tmax=2018 in 1500s.
# Let me push T_max a bit more cautious and just run it.

# But wait - 2 iters might fail at the lower-T zeros if input has only 38 sig digits. Let me also test 1 iter sometimes.
# Better: implement adaptive - terminate when difference becomes < target tolerance.

# Update runner to: max 4 iters, tolerance 1e-(target_dps+2), and use bracket-width sized appropriate.
# Current runner uses 1e-20 initial offset. That's bad for high-T zeros where adjacent zero is closer.
# Use offset = min(0.001, gap_to_neighbor/4) or just safe 1e-10 (zeros are far apart > 1e-5 minimum gap).
# Actually adjacent zeros at T=4075 separated by ~0.3, so 1e-10 offset is safe.

# Let me also try if smaller initial offset is fine since we already have 38-digit accuracy.
# A start offset of 1e-30 would be inside the cubic-convergence zone of Newton, but secant uses two starts.
# The 2-iter test above used 1e-30 offset - worked great.

# Update runner
new_runner = r'''#!/usr/bin/env python3
"""delta_shard_runner.py - hybrid search/refine runner for L(Delta,s) zeros.

Two modes:
 search: run lfunzeros at low precision (default 38) over [tmin, tmax],
 producing approximate zeros.
 refine: refine each approximate zero to dps=target via PARI lfunhardy +
 a 2-step secant iteration (sufficient because 38 sig digits initially
 quadratically expand: 38 -> ~61 -> ~99 digits).

Usage:
 python delta_shard_runner.py search <idx> <tmin> <tmax> <search_dps> <out_pkl>
 python delta_shard_runner.py refine <idx> <in_pkl> <target_dps> <out_pkl>
"""
import sys, time, pickle


def run_search(idx, tmin, tmax, search_dps, out_pkl):
 import cypari2
 pari = cypari2.Pari()
 pari.default("realprecision", search_dps)
 pari.allocatemem(10**9)
 pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")

 t0 = time.time()
 pari(f"Z = lfunzeros(L, [{tmin}, {tmax}])")
 n = int(pari("#Z"))
 elapsed = time.time() - t0

 fmt = f"%.{search_dps}g"
 approx = [str(pari(f'Strprintf("{fmt}", Z[{i}])')) for i in range(1, n + 1)]

 record = {
 "mode": "search",
 "idx": idx,
 "tmin": tmin,
 "tmax": tmax,
 "search_dps": search_dps,
 "elapsed": elapsed,
 "n": n,
 "approx_zeros": approx,
 }
 with open(out_pkl, "wb") as f:
 pickle.dump(record, f)
 print(f"[shard {idx} search] tmin={tmin} tmax={tmax} n={n} "
 f"elapsed={elapsed:.1f}s -> {out_pkl}", flush=True)


def run_refine(idx, in_pkl, target_dps, out_pkl):
 import cypari2
 with open(in_pkl, "rb") as f:
 record_in = pickle.load(f)
 approx = record_in["approx_zeros"]

 pari = cypari2.Pari()
 pari.default("realprecision", target_dps + 5)
 pari.allocatemem(2 * 10**9)
 pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")

 fmt = f"%.{target_dps}g"
 refined = []
 n_eval = 0
 t_total0 = time.time()
 for j, s in enumerate(approx):
 try:
 pari(f"t0 = {s} + 0.0")
 pari("t1 = t0 - 1e-30")
 pari("t2 = t0 + 1e-30")
 # 2 secant iterations (enough for 38 -> 80 digits)
 for it in range(2):
 pari("f1 = lfunhardy(L, t1)")
 pari("f2 = lfunhardy(L, t2)")
 pari("t3 = t2 - f2 * (t2 - t1) / (f2 - f1)")
 pari("t1 = t2")
 pari("t2 = t3")
 n_eval += 2
 # One more if diff still > 1e-(target+2)
 diff = float(pari("abs(t2 - t1)"))
 tol = 10 ** (-(target_dps + 2))
 tval = float(pari("abs(t2)")) or 1.0
 if diff > tol * tval:
 pari("f1 = lfunhardy(L, t1)")
 pari("f2 = lfunhardy(L, t2)")
 pari("t3 = t2 - f2 * (t2 - t1) / (f2 - f1)")
 pari("t1 = t2"); pari("t2 = t3")
 n_eval += 2
 t_refined = str(pari(f'Strprintf("{fmt}", t2)'))
 except Exception as e:
 print(f"[shard {idx} refine] FAIL j={j} s={s!r}: {e}", flush=True)
 t_refined = s
 refined.append(t_refined)
 if (j + 1) % 50 == 0:
 print(f" [shard {idx}] refined {j+1}/{len(approx)} "
 f"elapsed={time.time()-t_total0:.0f}s", flush=True)

 elapsed = time.time() - t_total0
 record = {
 "mode": "refine",
 "idx": idx,
 "target_dps": target_dps,
 "n": len(refined),
 "n_eval": n_eval,
 "elapsed": elapsed,
 "zeros": refined,
 "search_meta": {k: record_in[k] for k in ("tmin", "tmax", "search_dps")
 if k in record_in},
 }
 with open(out_pkl, "wb") as f:
 pickle.dump(record, f)
 print(f"[shard {idx} refine] n={len(refined)} n_eval={n_eval} "
 f"elapsed={elapsed:.1f}s -> {out_pkl}", flush=True)


def main():
 if len(sys.argv) < 2:
 print("usage: see file header", file=sys.stderr); sys.exit(2)
 mode = sys.argv[1]
 if mode == "search":
 if len(sys.argv) != 7:
 print("usage: ... search idx tmin tmax dps out_pkl", file=sys.stderr)
 sys.exit(2)
 _, _, idx, tmin, tmax, dps, out_pkl = sys.argv
 run_search(int(idx), float(tmin), float(tmax), int(dps), out_pkl)
 elif mode == "refine":
 if len(sys.argv) != 6:
 print("usage: ... refine idx in_pkl dps out_pkl", file=sys.stderr)
 sys.exit(2)
 _, _, idx, in_pkl, dps, out_pkl = sys.argv
 run_refine(int(idx), in_pkl, int(dps), out_pkl)
 else:
 print(f"unknown mode {mode!r}", file=sys.stderr); sys.exit(2)


if __name__ == "__main__":
 main()
'''
with open('work/delta_shard_runner.py', 'w') as f:
 f.write(new_runner)
print("wrote work/delta_shard_runner.py")

wrote work/delta_shard_runner.py


In [24]:
# Choose T_MAX so we can complete in our remaining budget. We've used ~10 min so far.
# Remaining budget ~3000s. Leave 500s for analysis/saving. Effective compute budget ~2500s.
# Plan: T_max=1900 -> ~ 2800 zeros, est total wall ~1400-1600s.
# 
# Let me update generate_ldelta.py with T_MAX=1900 for this run, but keep code structured 
# so T_MAX can be set via env var to support the original target of 4075.

new_driver = r'''#!/usr/bin/env python3
"""generate_ldelta.py - hybrid parallel generator for L(Delta,s) zeros.

Two-stage strategy:
 Stage 1 (search): partition [0, T_max] using cost-aware boundaries
 T_max * (i/N)**(1/(1+a)) with a=COST_EXP, run lfunzeros at
 SEARCH_DPS in NUM_SHARDS parallel subprocesses.
 Stage 2 (refine): for each shard's approximate-zero list, refine to TARGET_DPS
 via PARI lfunhardy + 2-step secant in NUM_SHARDS parallel
 subprocesses.

Final output: ldelta_zeros_N<Nactual>_dps80.npy (dtype='<U81').

Env vars (optional):
 LDELTA_TMAX T_MAX (default 4075.0)
 LDELTA_NTARGET N target (default 5000)
 LDELTA_SHARDS shard count (default 8)
 LDELTA_SEARCH_DPS search precision (default 38)
 LDELTA_TARGET_DPS target precision (default 80)
"""
import os, sys, subprocess, time, pickle
import numpy as np

T_MAX = float(os.environ.get("LDELTA_TMAX", "4075.0"))
SEARCH_DPS = int(os.environ.get("LDELTA_SEARCH_DPS", "38"))
TARGET_DPS = int(os.environ.get("LDELTA_TARGET_DPS", "80"))
N_TARGET = int(os.environ.get("LDELTA_NTARGET", "5000"))
NUM_SHARDS = int(os.environ.get("LDELTA_SHARDS", "8"))
COST_EXP = 2.85 # empirical lfunzeros cost scaling

SHARD_DIR = os.environ.get("LDELTA_SHARD_DIR", "ldelta_shards")
FINAL_NPY_TEMPLATE = "ldelta_zeros_N{N}_dps{dps}.npy"

GAMMA1_REF = ("9.22237939992110252224376719274347813552877062243200928999818642"
 "800192129537")


def partition_costaware(T_max, n, a):
 """Boundaries balancing integral of T^a from 0 to T_max across n shards."""
 return [T_max * (i / n) ** (1.0 / (a + 1.0)) for i in range(n + 1)]


def launch(cmd, log_path):
 lf = open(log_path, "w")
 print(" $", " ".join(cmd))
 p = subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT)
 return p, lf


def main():
 print(f"T_MAX={T_MAX} SEARCH_DPS={SEARCH_DPS} TARGET_DPS={TARGET_DPS} "
 f"N_TARGET={N_TARGET} NUM_SHARDS={NUM_SHARDS} COST_EXP={COST_EXP}",
 flush=True)
 os.makedirs(SHARD_DIR, exist_ok=True)
 edges = partition_costaware(T_MAX, NUM_SHARDS, COST_EXP)
 print(f"Cost-aware partition (a={COST_EXP}) edges:")
 for i, e in enumerate(edges):
 print(f" edge[{i}] = {e:.6f}")
 print(flush=True)

 here = os.path.dirname(os.path.abspath(__file__))
 runner = os.path.join(here, "delta_shard_runner.py")

 # ---------- Stage 1: search ----------
 print(f"=== Stage 1: search (dps={SEARCH_DPS}) ===", flush=True)
 procs = []
 search_pkls = []
 t1 = time.time()
 for i in range(NUM_SHARDS):
 tmin, tmax = edges[i], edges[i + 1]
 out_pkl = os.path.join(SHARD_DIR, f"ldelta_search_{i}.pkl")
 log_path = os.path.join(SHARD_DIR, f"search_{i}.log")
 search_pkls.append(out_pkl)
 cmd = [sys.executable, runner, "search", str(i),
 repr(tmin), repr(tmax), str(SEARCH_DPS), out_pkl]
 print(f" shard {i}: [{tmin:.4f}, {tmax:.4f}]")
 p, lf = launch(cmd, log_path)
 procs.append((p, lf, i))
 rc = []
 for p, lf, i in procs:
 ret = p.wait(); lf.close(); rc.append(ret)
 print(f" search shard {i} rc={ret} elapsed={time.time()-t1:.1f}s",
 flush=True)
 if any(r != 0 for r in rc):
 print("ERROR: search stage failed", file=sys.stderr); sys.exit(1)
 t_stage1 = time.time() - t1
 # Count zeros found per shard
 counts = []
 for pkl in search_pkls:
 with open(pkl, "rb") as f: r = pickle.load(f)
 counts.append(r["n"])
 print(f"Stage 1 wall time: {t_stage1:.1f}s; zeros/shard={counts} "
 f"(total {sum(counts)})\n", flush=True)

 # ---------- Stage 2: refine ----------
 print(f"=== Stage 2: refine (dps={TARGET_DPS}) ===", flush=True)
 procs = []
 refine_pkls = []
 t2 = time.time()
 for i in range(NUM_SHARDS):
 in_pkl = search_pkls[i]
 out_pkl = os.path.join(SHARD_DIR, f"ldelta_refine_{i}.pkl")
 log_path = os.path.join(SHARD_DIR, f"refine_{i}.log")
 refine_pkls.append(out_pkl)
 cmd = [sys.executable, runner, "refine", str(i), in_pkl,
 str(TARGET_DPS), out_pkl]
 p, lf = launch(cmd, log_path)
 procs.append((p, lf, i))
 rc = []
 for p, lf, i in procs:
 ret = p.wait(); lf.close(); rc.append(ret)
 print(f" refine shard {i} rc={ret} elapsed={time.time()-t2:.1f}s",
 flush=True)
 if any(r != 0 for r in rc):
 print("ERROR: refine stage failed", file=sys.stderr); sys.exit(1)
 t_stage2 = time.time() - t2
 print(f"Stage 2 wall time: {t_stage2:.1f}s\n", flush=True)

 # ---------- Merge / sort / dedup / save ----------
 all_zeros = []
 for i, pkl in enumerate(refine_pkls):
 with open(pkl, "rb") as f: r = pickle.load(f)
 print(f" refine shard {i}: n={r['n']} elapsed={r['elapsed']:.1f}s")
 all_zeros.extend(r["zeros"])
 print(f"\nTotal zeros collected (raw): {len(all_zeros)}")

 pairs = []
 for s in all_zeros:
 try:
 v = float(s)
 except Exception as e:
 print(f" WARN unparseable {s!r}: {e}"); continue
 pairs.append((v, s))
 pairs.sort(key=lambda x: x[0])
 dedup = []
 prev_v = None
 TOL = 1e-9
 for v, s in pairs:
 if prev_v is not None and abs(v - prev_v) < TOL:
 continue
 dedup.append((v, s)); prev_v = v
 print(f"After de-duplication: {len(dedup)}")

 N_actual = min(N_TARGET, len(dedup))
 trimmed = [s for (_, s) in dedup[:N_actual]]
 if N_actual < N_TARGET:
 print(f"NOTE: produced N={N_actual} < N_TARGET={N_TARGET}", flush=True)

 g1 = trimmed[0] if trimmed else ""
 print(f"\ngamma_1 (computed): {g1}")
 print(f"gamma_1 (reference): {GAMMA1_REF}")
 nmatch = 0
 for a_, b_ in zip(g1, GAMMA1_REF):
 if a_ == b_: nmatch += 1
 else: break
 print(f"matching leading characters: {nmatch}")

 out_npy = FINAL_NPY_TEMPLATE.format(N=N_actual, dps=TARGET_DPS)
 arr = np.array(trimmed, dtype="<U81")
 np.save(out_npy, arr)
 print(f"\nSaved {arr.shape[0]} zeros to {out_npy} (dtype={arr.dtype})")
 if arr.shape[0]:
 print(f"First zero: {arr[0]}")
 print(f"Last zero: {arr[-1]}")
 print(f"Total wall (stages 1+2): {t_stage1 + t_stage2:.1f}s")


if __name__ == "__main__":
 main()
'''
with open('work/generate_ldelta.py', 'w') as f:
 f.write(new_driver)
print("wrote work/generate_ldelta.py")

wrote work/generate_ldelta.py


In [25]:
# Let's first do a small calibration run to validate the architecture works end-to-end. T_max=300 should produce ~150 zeros and take a few seconds.
import os, subprocess, sys, time
env = os.environ.copy()
env['LDELTA_TMAX'] = '300'
env['LDELTA_NTARGET'] = '5000' # we won't reach it but that's fine
env['LDELTA_SHARD_DIR'] = 'ldelta_shards_test'
os.chdir('/workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work')
t0=time.time()
r = subprocess.run([sys.executable, 'generate_ldelta.py'], capture_output=True, text=True, env=env, timeout=600)
print(f"wall: {time.time()-t0:.1f}s")
print("OUT:", r.stdout)
print("ERR:", r.stderr[-2000:])

wall: 7.0s
OUT: T_MAX=300.0 SEARCH_DPS=38 TARGET_DPS=80 N_TARGET=5000 NUM_SHARDS=8 COST_EXP=2.85
Cost-aware partition (a=2.85) edges:
 edge[0] = 0.000000
 edge[1] = 174.804427
 edge[2] = 209.286896
 edge[3] = 232.530554
 edge[4] = 250.571484
 edge[5] = 265.523569
 edge[6] = 278.400259
 edge[7] = 289.773330
 edge[8] = 300.000000

=== Stage 1: search (dps=38) ===
 shard 0: [0.0000, 174.8044]
 $ /app/kernel_env/bin/python /workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work/delta_shard_runner.py search 0 0.0 174.80442691572728 38 ldelta_shards_test/ldelta_search_0.pkl
 shard 1: [174.8044, 209.2869]
 $ /app/kernel_env/bin/python /workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work/delta_shard_runner.py search 1 174.80442691572728 209.28689567061753 38 ldelta_shards_test/ldelta_search_1.pkl
 shard 2: [209.2869, 232.5306]
 $ /app/kernel_env/bin/python /workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work/delta_shard_runner.py search 2 209.28689567061753 232.53055359798202 38 ldelta_shards_tes

In [26]:
# PERFECT. End-to-end: T_max=300 -> 277 zeros, dps=80, gamma_1 matches reference to all 76 ref digits.
# Total wall time 6.8s (5.9s refine, ~1s search). 
# 
# Notes:
# - The cost-aware partition for low T_max is unbalanced because the integrand T^2.85 is dominated 
# by the upper end. Shard 0 has 132 zeros vs shard 7 only 13. Wall time still balanced (5.9s each), 
# so the partition is properly accounting for cost.
# - The refine step took ~3.3-5.7s per shard at low T - dominated by PARI startup overhead!
# This means our extrapolation for stage2 overcounted startup. Each refine eval cost grows with T but
# for small T we have fixed startup of ~2-3s.
# 
# Now let me make a moderately-sized run for the actual N=5000 attempt. 
# With remaining ~2500s budget, we can do T_max ~ 1900-2200.
# Try T_max=2000 first.

# Cleanup test
import shutil
shutil.rmtree('ldelta_shards_test', ignore_errors=True)
os.remove('ldelta_zeros_N277_dps80.npy')
print("ok")

ok


In [27]:
# Run full attempt. Set T_MAX so we maximize zeros produced within budget.
# Aim ~2200s wall time, with margin. Estimate: T_max=2200 gives ~3500 zeros, total wall ~2400s based on extrapolation.
# Let me be more conservative: T_max=2100.

import os, subprocess, sys, time
env = os.environ.copy()
env['LDELTA_TMAX'] = '2200'
env['LDELTA_NTARGET'] = '5000'
env['LDELTA_SHARD_DIR'] = 'ldelta_shards'

os.chdir('/workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work')

# Stream output by launching as a subprocess with live logging
import threading, queue
log_path = 'run_log.txt'
t0 = time.time()
p = subprocess.Popen([sys.executable, 'generate_ldelta.py'],
 stdout=open(log_path,'w'), stderr=subprocess.STDOUT, env=env)
# Poll periodically
print(f"Launched PID {p.pid}, T_MAX=2200")
# Wait up to 2400s
DEADLINE = 2300
while True:
 elapsed = time.time() - t0
 rc = p.poll()
 if rc is not None:
 print(f"Process finished, rc={rc}, elapsed={elapsed:.1f}s")
 break
 if elapsed > DEADLINE:
 print(f"DEADLINE reached at {elapsed:.0f}s; terminating")
 p.terminate()
 try:
 p.wait(timeout=30)
 except subprocess.TimeoutExpired:
 p.kill()
 break
 time.sleep(20)
 # print tail of log
 try:
 with open(log_path) as f:
 tail = f.read().splitlines()
 print(f"[t={elapsed:.0f}s] " + " | ".join(tail[-3:]))
 except Exception:
 pass

TimeoutError: Code execution timed out after 900.0 seconds

In [28]:
# Cell timeout but the subprocess is still running. Let me check status and tail.
import subprocess, os, time
log_path = '/workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work/run_log.txt'
# Find the running process
r = subprocess.run(['pgrep', '-af', 'generate_ldelta.py'], capture_output=True, text=True)
print("Driver process:", r.stdout)
r = subprocess.run(['pgrep', '-af', 'delta_shard_runner.py'], capture_output=True, text=True)
print("Shard processes:")
print(r.stdout)
print("---log tail---")
with open(log_path) as f:
 lines = f.read().splitlines()
for ln in lines[-30:]:
 print(ln)

PermissionError: [Errno 13] Permission denied: 'pgrep'

In [29]:
import subprocess, os
# Use /proc to find python processes
import glob, os
my_pid = os.getpid()
candidates = []
for pid_dir in glob.glob('/proc/[0-9]*'):
 pid = int(pid_dir.split('/')[-1])
 if pid == my_pid: continue
 try:
 with open(f'{pid_dir}/cmdline') as f:
 cmd = f.read().replace('\x00', ' ').strip()
 if 'generate_ldelta' in cmd or 'delta_shard_runner' in cmd:
 try:
 with open(f'{pid_dir}/stat') as f:
 stat = f.read()
 candidates.append((pid, stat.split()[2], cmd))
 except:
 candidates.append((pid, '?', cmd))
 except Exception:
 pass
for pid, state, cmd in candidates:
 print(f"PID {pid} state={state}: {cmd[:120]}")

log_path = '/workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work/run_log.txt'
print("---log tail---")
with open(log_path) as f:
 lines = f.read().splitlines()
for ln in lines[-25:]:
 print(ln)

---log tail---
=== Stage 2: refine (dps=80) ===
 $ /app/kernel_env/bin/python /workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work/delta_shard_runner.py refine 0 ldelta_shards/ldelta_search_0.pkl 80 ldelta_shards/ldelta_refine_0.pkl
 $ /app/kernel_env/bin/python /workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work/delta_shard_runner.py refine 1 ldelta_shards/ldelta_search_1.pkl 80 ldelta_shards/ldelta_refine_1.pkl
 $ /app/kernel_env/bin/python /workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work/delta_shard_runner.py refine 2 ldelta_shards/ldelta_search_2.pkl 80 ldelta_shards/ldelta_refine_2.pkl
 $ /app/kernel_env/bin/python /workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work/delta_shard_runner.py refine 3 ldelta_shards/ldelta_search_3.pkl 80 ldelta_shards/ldelta_refine_3.pkl
 $ /app/kernel_env/bin/python /workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work/delta_shard_runner.py refine 4 ldelta_shards/ldelta_search_4.pkl 80 ldelta_shards/ldelta_refine_4.pkl
 $ /app/kernel_env/bin/pyth

In [30]:
# The driver got KeyboardInterrupt (from our cell timeout) so it exited. But the shard subprocesses 
# may still be running. Check.
import glob, os
my_pid = os.getpid()
candidates = []
for pid_dir in glob.glob('/proc/[0-9]*'):
 try:
 pid = int(pid_dir.split('/')[-1])
 except: continue
 if pid == my_pid: continue
 try:
 with open(f'{pid_dir}/cmdline') as f:
 cmd = f.read().replace('\x00', ' ').strip()
 if 'delta_shard_runner' in cmd or 'generate_ldelta' in cmd:
 with open(f'{pid_dir}/stat') as f:
 stat = f.read()
 candidates.append((pid, stat.split()[2], cmd))
 except Exception:
 pass
for pid, state, cmd in candidates:
 print(f"PID {pid} state={state}: ...{cmd[-100:]}")

In [31]:
# No processes remain - everything got cleaned up.
# Check what we have: did stage 1 finish? Look at search pickle files.
import os, pickle
sd = '/workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work/ldelta_shards'
print(os.listdir(sd))

['search_2.log', 'ldelta_search_5.pkl', 'search_6.log', 'search_3.log', 'refine_2.log', 'ldelta_search_3.pkl', 'refine_6.log', 'search_0.log', 'search_7.log', 'ldelta_search_1.pkl', 'ldelta_search_4.pkl', 'refine_5.log', 'ldelta_search_2.pkl', 'refine_7.log', 'ldelta_search_7.pkl', 'refine_3.log', 'ldelta_search_6.pkl', 'search_5.log', 'search_4.log', 'refine_0.log', 'refine_1.log', 'ldelta_search_0.pkl', 'refine_4.log', 'search_1.log']


In [32]:
# Stage 1 (search) completed - all 8 search pickle files exist.
# Stage 2 (refine) did not produce any pickle files - was killed early.
import pickle
for i in range(8):
 with open(f'{sd}/ldelta_search_{i}.pkl', 'rb') as f:
 r = pickle.load(f)
 print(f"search {i}: tmin={r['tmin']:.2f} tmax={r['tmax']:.2f} n={r['n']} elapsed={r['elapsed']:.1f}s")
# also check refine logs to see how far it got
print("\n--- refine logs (tails) ---")
for i in range(8):
 with open(f'{sd}/refine_{i}.log') as f:
 lines = f.read().splitlines()
 print(f"refine {i}: ", " | ".join(lines[-2:]) if lines else "(empty)")

search 0: tmin=0.00 tmax=1281.90 n=1764 elapsed=400.5s
search 1: tmin=1281.90 tmax=1534.77 n=436 elapsed=349.3s
search 2: tmin=1534.77 tmax=1705.22 n=302 elapsed=314.8s
search 3: tmin=1705.22 tmax=1837.52 n=237 elapsed=299.3s
search 4: tmin=1837.52 tmax=1947.17 n=199 elapsed=298.2s
search 5: tmin=1947.17 tmax=2041.60 n=174 elapsed=301.4s
search 6: tmin=2041.60 tmax=2125.00 n=154 elapsed=300.7s
search 7: tmin=2125.00 tmax=2200.00 n=139 elapsed=298.3s

--- refine logs (tails) ---
refine 0: File "cypari2/convert.pyx", line 545, in cypari2.convert.PyObject_AsGEN | KeyboardInterrupt
refine 1: File "cypari2/convert.pyx", line 545, in cypari2.convert.PyObject_AsGEN | KeyboardInterrupt
refine 2: File "cypari2/convert.pyx", line 545, in cypari2.convert.PyObject_AsGEN | KeyboardInterrupt
refine 3: File "cypari2/convert.pyx", line 545, in cypari2.convert.PyObject_AsGEN | KeyboardInterrupt
refine 4: File "cypari2/convert.pyx", line 545, in cypari2.convert.PyObject_AsGEN | KeyboardInterrupt
refine 

In [33]:
# GREAT NEWS:
# - Stage 1 search completed in ~400s wall time (max shard 0 = 400s) - WELL within budget.
# - Total zeros found in stage 1: 1764 + 436 + 302 + 237 + 199 + 174 + 154 + 139 = 3405 zeros up to T=2200.
# That's fewer than we predicted (we expected ~3066 from N(T)≈3066 at T=2018, but lfunzeros found more in [0, 2200])
# - Cost-aware partition WORKS: shard wall times are all ~298-400s, while zeros are wildly imbalanced (1764 vs 139).
# This validates that T^2.85 cost scaling correctly captures the per-zero work growth.
# - Stage 2 was killed by our kernel timeout before producing anything.
# 
# Now: we have search pickles for 3405 approximate zeros. Need to refine. 
# Time budget check: how much time elapsed so far?
import time
# Refine cost: integrate dN/dT * 4 * 4.34e-9 * T^2.68 dT from 0 to 2200, divided by 8
import math
from scipy.integrate import quad
def dN_dT(T):
 return math.log(T/(2*math.pi))/math.pi
def cost2(T):
 return dN_dT(T) * 4 * 4.34e-9 * T**2.68
est_total_cpu, _ = quad(cost2, 100, 2200)
print(f"Stage 2 est total CPU-s: {est_total_cpu:.0f}")
print(f"Stage 2 est wall (8 cores, perfect balance): {est_total_cpu/8:.0f}s")

# Per-shard work distribution: each shard refines zeros in its T-range. Stage 1 was balanced for SEARCH cost
# (T^2.85), but stage 2 work scales as T^2.68 - similar but not identical. The shards with high-T zeros 
# will dominate stage 2 time.
# Estimate per-shard stage 2 wall time:
shard_ranges = [(0, 1281.90), (1281.90, 1534.77), (1534.77, 1705.22), (1705.22, 1837.52),
 (1837.52, 1947.17), (1947.17, 2041.60), (2041.60, 2125.00), (2125.00, 2200.00)]
shard_counts = [1764, 436, 302, 237, 199, 174, 154, 139]
for i, ((tlo, thi), nz) in enumerate(zip(shard_ranges, shard_counts)):
 est = quad(cost2, max(tlo, 1), thi)[0]
 print(f"shard {i} range=[{tlo:.0f},{thi:.0f}] nz={nz} est_wall={est:.0f}s")

Stage 2 est total CPU-s: 16742
Stage 2 est wall (8 cores, perfect balance): 2093s
shard 0 range=[0,1282] nz=1764 est_wall=2072s
shard 1 range=[1282,1535] nz=436 est_wall=2091s
shard 2 range=[1535,1705] nz=302 est_wall=2094s
shard 3 range=[1705,1838] nz=237 est_wall=2096s
shard 4 range=[1838,1947] nz=199 est_wall=2097s
shard 5 range=[1947,2042] nz=174 est_wall=2097s
shard 6 range=[2042,2125] nz=154 est_wall=2097s
shard 7 range=[2125,2200] nz=139 est_wall=2098s


In [34]:
# Cost-aware partition has perfectly balanced predicted stage-2 wall times too (~2090s each), 
# because the cost integrand T^2.68 is very close to T^2.85.
# 
# Estimated stage 2 = ~2090s per shard (in parallel). We've used ~17 min of cell time so far in this session.
# This is going to be very tight. Let me check: the cell timeout is 900s but we can run a single subprocess detached.
# Better: launch stage 2 directly (skipping the rerun of stage 1) and let it run while we monitor.
# Time so far: 7 min calibration + 15 min stage1 + few min interactive = ~25 min. Remaining: 35 min = 2100s.
# 
# So we have only ~2100s left for stage 2. The estimate says 2090s for full stage 2 on perfect balance.
# This is going to be marginal. Let me start it now.

# Write a small script to run only stage 2 from the existing search pickles
stage2_only = r'''#!/usr/bin/env python3
"""stage2_only.py - run only the refine stage from existing search pickles."""
import os, sys, subprocess, time, pickle
import numpy as np

NUM_SHARDS = 8
TARGET_DPS = 80
N_TARGET = 5000
SHARD_DIR = "ldelta_shards"

here = os.path.dirname(os.path.abspath(__file__))
runner = os.path.join(here, "delta_shard_runner.py")

print("=== Stage 2: refine ===", flush=True)
procs = []
refine_pkls = []
t2 = time.time()
for i in range(NUM_SHARDS):
 in_pkl = os.path.join(SHARD_DIR, f"ldelta_search_{i}.pkl")
 out_pkl = os.path.join(SHARD_DIR, f"ldelta_refine_{i}.pkl")
 log_path = os.path.join(SHARD_DIR, f"refine_{i}.log")
 refine_pkls.append(out_pkl)
 cmd = [sys.executable, runner, "refine", str(i), in_pkl, str(TARGET_DPS), out_pkl]
 print(f" shard {i}: {in_pkl}", flush=True)
 lf = open(log_path, "w")
 p = subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT)
 procs.append((p, lf, i))

rc = []
for p, lf, i in procs:
 ret = p.wait(); lf.close(); rc.append(ret)
 print(f" refine shard {i} rc={ret} elapsed={time.time()-t2:.1f}s", flush=True)
t_stage2 = time.time() - t2

# Merge
all_zeros = []
for i, pkl in enumerate(refine_pkls):
 if not os.path.exists(pkl):
 print(f" WARNING: missing {pkl}")
 continue
 with open(pkl, "rb") as f: r = pickle.load(f)
 print(f" refine shard {i}: n={r['n']} elapsed={r['elapsed']:.1f}s")
 all_zeros.extend(r["zeros"])
print(f"\nTotal raw: {len(all_zeros)}")

pairs = []
for s in all_zeros:
 try:
 v = float(s)
 except Exception:
 continue
 pairs.append((v, s))
pairs.sort(key=lambda x: x[0])
dedup, prev = [], None
for v, s in pairs:
 if prev is not None and abs(v - prev) < 1e-9: continue
 dedup.append((v, s)); prev = v
print(f"After dedup: {len(dedup)}")

trimmed = [s for (_, s) in dedup[:N_TARGET]]
N_actual = len(trimmed)
print(f"N_actual: {N_actual}")

GAMMA1_REF = ("9.22237939992110252224376719274347813552877062243200928999818642"
 "800192129537")
if trimmed:
 g1 = trimmed[0]
 nm = 0
 for a_, b_ in zip(g1, GAMMA1_REF):
 if a_ == b_: nm += 1
 else: break
 print(f"gamma_1 matches first {nm} chars of reference")

out_npy = f"ldelta_zeros_N{N_actual}_dps{TARGET_DPS}.npy"
arr = np.array(trimmed, dtype="<U81")
np.save(out_npy, arr)
print(f"Saved to {out_npy} (shape={arr.shape}, dtype={arr.dtype})")
print(f"Stage 2 wall: {t_stage2:.1f}s")
'''
with open('/workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work/stage2_only.py', 'w') as f:
 f.write(stage2_only)
print("wrote stage2_only.py")

wrote stage2_only.py


In [35]:
# Launch stage 2 in fully detached mode (so kernel cell timeout won't kill it)
# Use nohup-style detachment via setsid in Python
import subprocess, os, sys, time

os.chdir('/workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work')
log = open('stage2_log.txt', 'w')
# Launch with new process group so it survives kernel cell timeouts
p = subprocess.Popen([sys.executable, 'stage2_only.py'],
 stdout=log, stderr=subprocess.STDOUT,
 start_new_session=True) # detach from kernel session
print(f"launched stage2_only.py as PID {p.pid}, started_at={time.time():.0f}")
print(f"detached: True")
# Don't wait. Save PID to file for later checking.
with open('stage2.pid', 'w') as f:
 f.write(str(p.pid))

launched stage2_only.py as PID 173, started_at=1780180520
detached: True


In [36]:
# Poll without holding the cell open
import time, os, glob
def shard_status():
 sd = '/workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work/ldelta_shards'
 status = []
 for i in range(8):
 log = f'{sd}/refine_{i}.log'
 pkl = f'{sd}/ldelta_refine_{i}.pkl'
 try:
 with open(log) as f: lines = f.read().splitlines()
 last = lines[-1] if lines else '(empty)'
 except: last = '(no log)'
 done = os.path.exists(pkl)
 status.append((i, done, last))
 return status

def is_alive(pid):
 try:
 os.kill(int(pid), 0)
 return True
 except OSError:
 return False

with open('/workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work/stage2.pid') as f:
 pid = f.read().strip()
print(f"PID {pid} alive: {is_alive(pid)}")
for i, done, last in shard_status():
 mark = '✓' if done else '·'
 print(f" {mark} shard {i}: {last[:100]}")

PID 173 alive: True
 · shard 0: [shard 0] refined 100/1764 elapsed=4s
 · shard 1: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
 · shard 2: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
 · shard 3: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
 · shard 4: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
 · shard 5: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
 · shard 6: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
 · shard 7: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072


In [37]:
# Shard 0 refined 100/1764 zeros in 4s = very fast since low-T zeros (<100 dps). 
# Let me wait a bit and check progress
import time
time.sleep(60)
print(f"checking after 60s sleep")
for i, done, last in shard_status():
 mark = '✓' if done else '·'
 print(f" {mark} shard {i}: {last[:120]}")
print(f"\nPID {pid} alive: {is_alive(pid)}")

checking after 60s sleep
 · shard 0: [shard 0] refined 400/1764 elapsed=78s
 · shard 1: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
 · shard 2: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
 · shard 3: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
 · shard 4: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
 · shard 5: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
 · shard 6: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
 · shard 7: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072

PID 173 alive: True


In [38]:
# Other shards haven't printed progress yet (they print every 50 - and shard 1's 436 zeros at high T are slower).
# Shard 0 doing well: 400/1764 in 78s -> projected ~344s total for shard 0.
# Wait some more.
import time
time.sleep(180)
print(f"checking after 180s more")
for i, done, last in shard_status():
 mark = '✓' if done else '·'
 print(f" {mark} shard {i}: {last[:140]}")
print(f"\nPID {pid} alive: {is_alive(pid)}")

checking after 180s more
 · shard 0: [shard 0] refined 600/1764 elapsed=219s
 · shard 1: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
 · shard 2: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
 · shard 3: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
 · shard 4: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
 · shard 5: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
 · shard 6: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
 · shard 7: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072

PID 173 alive: True


In [39]:
# Shard 0 progress is slowing: 100/4s -> 400/78s (200/74s) -> 600/219s (200/141s). 
# That's because zeros get harder as T grows. Shard 0 covers T=0..1281.
# Extrapolating: time per 200 batch at zeros 600-800 will be ~200-250s, etc.
# Other shards have only printed startup line but are silently working on high-T zeros - 
# each one taking ~5-30s currently. They print every 50. So shard 1-7 haven't reached 50 yet.
# 
# Shard 1 has 436 zeros at T=1282-1535. At T=1500, hardy@dps85 = ~1.5s, so 4 evals = 6s/zero -> ~2616s total. 
# Shard 7 has 139 zeros at T=2125-2200. At T=2200, hardy ~ 3s, so 12s/zero -> 1668s total.
# So shard 1 is the bottleneck at ~2616s.

# Already used ~6 min stage2. Remaining budget tight. Let me check elapsed total.
import time
print(f"current cell time...")
# Let me figure out total budget remaining
# We started around the beginning. Approx 30 min into session at stage 1 launch. 
# Now ~30+15+6 = 51 min in. Budget 60 min. ~9 min left.
# Stage 2 needs ~30-40 more minutes for shard 1. 
# Will not finish in budget.

# Strategy: 
# - Document what we've accomplished
# - Capture whatever is complete when budget runs out
# - The architecture is validated; key inadequacy is that we don't have enough time even with the optimizations

# Continue waiting for as much progress as possible.
time.sleep(120)
for i, done, last in shard_status():
 mark = '✓' if done else '·'
 print(f" {mark} shard {i}: {last[:140]}")

current cell time...


 · shard 0: [shard 0] refined 750/1764 elapsed=399s
 · shard 1: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
 · shard 2: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
 · shard 3: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
 · shard 4: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
 · shard 5: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
 · shard 6: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
 · shard 7: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072


In [40]:
# Shard 0: 750/1764 at 399s. task: 0->100/4s, ->400/78s, ->600/219s, ->750/399s.
# So later batches taking ~180s/150 zeros (1.2s/zero average at T~700-900)
# Remaining: 1014 zeros at T=900-1281; per zero will hit ~3-5s. Total to finish shard 0 maybe ~3000s more. Way too long.
# 
# Other shards working silently. They will reach 50 marker eventually.
# Given we have only ~7 min budget left, decision: terminate stage 2 NOW and save partial state.
# Implement a "rescue" mechanism: stop the running processes, collect any partially-completed work, 
# combine with un-refined approx zeros, save what we have.
# 
# Actually: the refine subprocesses only write their pkl at the very end. We won't get any refined output 
# unless we modify them to write incrementally.
# 
# Plan: kill stage 2, save the SEARCH-only zeros (dps=38) as the partial best-effort output (clearly labeled).
# Then in final analysis we'll honestly report what we accomplished.

import os, signal
# kill the stage 2 driver and all shard runners
with open('/workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work/stage2.pid') as f:
 pid = int(f.read().strip())
try:
 os.killpg(pid, signal.SIGTERM)
 print(f"killed PG {pid}")
except Exception as e:
 print(f"failed to kill PG: {e}")
 try: os.kill(pid, signal.SIGTERM); print(f"killed PID {pid}")
 except Exception as ee: print(f"kill failed: {ee}")

import time
time.sleep(3)
# Also kill any straggler shard runners
import glob
for pid_dir in glob.glob('/proc/[0-9]*'):
 try:
 spid = int(pid_dir.split('/')[-1])
 with open(f'{pid_dir}/cmdline') as f:
 cmd = f.read().replace('\x00', ' ')
 if 'delta_shard_runner' in cmd:
 try: os.kill(spid, signal.SIGTERM); print(f" killed shard runner pid {spid}")
 except: pass
 except: pass

killed PG 173


In [41]:
# Let me at least save shard 0's partial progress. Wait - the shard writes pkl only at end.
# But we have the search pkls with dps=38 approximations for all 3405 zeros. Let me save THOSE
# clearly labeled as approximations, and produce a partial dps=80 file with whatever refined zeros we have.

# Actually shard 0 reached 750+ refined zeros at T<=~900. But it didn't dump its pkl. We can't easily recover.
# 
# Better plan: do a fast refinement on the small T zeros (cheap) ourselves, right here, in time we have remaining.
# At T<500, hardy at dps85 is <0.1s, so 4 evals = 0.4s/zero. For all zeros up to T=500 (~500 zeros), 
# would take ~200s on 1 cpu. We could do this in this cell.

# Even better idea: many zeros at LOW T have refinement essentially free. Let me do this efficiently:
# Run one PARI session, refine zeros up to some T cutoff that we can finish quickly.

import subprocess, sys, time, pickle, os

# Gather all approx zeros from all 8 search pickles
sd = '/workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work/ldelta_shards'
all_approx = []
for i in range(8):
 with open(f'{sd}/ldelta_search_{i}.pkl', 'rb') as f:
 r = pickle.load(f)
 all_approx.extend(r['approx_zeros'])
print(f"Total approx zeros from search: {len(all_approx)}")
# Sort numerically
all_approx_pairs = sorted(((float(s), s) for s in all_approx), key=lambda x: x[0])
# Save for archival
with open('/workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work/ldelta_approx_dps38_all.pkl', 'wb') as f:
 pickle.dump([s for v, s in all_approx_pairs], f)
print(f"first zero: {all_approx_pairs[0][1]}")
print(f"last zero: {all_approx_pairs[-1][1]}")
# How many up to T=500?
n500 = sum(1 for v,_ in all_approx_pairs if v <= 500)
n1000 = sum(1 for v,_ in all_approx_pairs if v <= 1000)
print(f"up to T=500: {n500}, T=1000: {n1000}")

Total approx zeros from search: 3405
first zero: 9.2223793999211025222437671927434781355
last zero: 2199.5340780429643642638069007845685866
up to T=500: 540, T=1000: 1298


In [42]:
# Already have 1298 zeros up to T=1000. Note: this matches the existing ldelta_zeros_N1298_dps80.npy mentioned!
# So we're recovering known territory. Need to refine 3405 approx zeros to dps=80 quickly.
# 
# Idea: launch 8 parallel quick-refine subprocesses, each handling a chunk. Each writes incrementally? No, our 
# refine code waits till end.
# 
# Let me modify the runner to write incrementally (save every 50 zeros). Then re-launch with a strict time budget.
# We have ~7 min budget. Realistically:
# - Refining all 1298 zeros up to T=1000 takes about: at T=1000 hardy is 0.46s -> 1.8s/zero -> avg ~0.5s/zero ->
# ~650s on 1 cpu. With 8 cpus split ~80s. That's fast!
# - Refining all 3405 zeros up to T=2200 would need ~ 8 * 1.5 * 4 evals avg = some unfortunate amount.

# Let me make an incremental version of the runner. Time is short - 5 min cap to set up + run.
incr_runner = r'''#!/usr/bin/env python3
"""quick_refine.py - refine a list of approx zeros, writing output incrementally.

Usage: quick_refine.py <idx> <in_pkl_listzeros> <target_dps> <out_pkl> [<deadline_seconds>]
The in_pkl is a pickled list of strings.
"""
import sys, time, pickle, os

idx = int(sys.argv[1])
in_pkl = sys.argv[2]
target_dps = int(sys.argv[3])
out_pkl = sys.argv[4]
deadline = float(sys.argv[5]) if len(sys.argv) > 5 else 1e18

with open(in_pkl, 'rb') as f:
 approx = pickle.load(f)

import cypari2
pari = cypari2.Pari()
pari.default("realprecision", target_dps + 5)
pari.allocatemem(2 * 10**9)
pari("L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])")

fmt = f"%.{target_dps}g"
refined = []
n_eval = 0
t0 = time.time()
N = len(approx)
for j, s in enumerate(approx):
 if time.time() - t0 > deadline:
 print(f"[{idx}] DEADLINE at j={j}/{N}", flush=True)
 break
 try:
 pari(f"t0 = {s} + 0.0")
 pari("t1 = t0 - 1e-30"); pari("t2 = t0 + 1e-30")
 for it in range(2):
 pari("f1 = lfunhardy(L, t1)"); pari("f2 = lfunhardy(L, t2)")
 pari("t3 = t2 - f2 * (t2 - t1) / (f2 - f1)")
 pari("t1 = t2"); pari("t2 = t3")
 n_eval += 2
 r = str(pari(f'Strprintf("{fmt}", t2)'))
 except Exception as e:
 print(f"[{idx}] fail j={j}: {e}", flush=True)
 r = s
 refined.append(r)
 # incremental dump every 25 zeros
 if (j+1) % 25 == 0:
 with open(out_pkl, 'wb') as f:
 pickle.dump({'mode':'refine','idx':idx,'n':len(refined),
 'zeros':refined,'n_eval':n_eval,
 'elapsed':time.time()-t0,'completed':False}, f)
 print(f"[{idx}] {j+1}/{N} elapsed={time.time()-t0:.0f}s", flush=True)

with open(out_pkl, 'wb') as f:
 pickle.dump({'mode':'refine','idx':idx,'n':len(refined),
 'zeros':refined,'n_eval':n_eval,
 'elapsed':time.time()-t0,'completed':True}, f)
print(f"[{idx}] DONE n={len(refined)}/{N} elapsed={time.time()-t0:.0f}s", flush=True)
'''
with open('/workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work/quick_refine.py', 'w') as f:
 f.write(incr_runner)
print("wrote quick_refine.py")

wrote quick_refine.py


In [43]:
# Partition the 3405 zeros into 8 equal-COST chunks. Cost per zero ~ T^2.68 in time.
# Compute cumulative cost weights and split into 8 equal chunks.

import pickle, math
all_zeros_sorted = [s for v, s in all_approx_pairs]
all_T = [v for v, s in all_approx_pairs]
weights = [max(T,1)**2.68 for T in all_T]
cum = []
c = 0
for w in weights:
 c += w
 cum.append(c)
total = cum[-1]
boundaries = []
for i in range(1, 8):
 target = total * i / 8
 # find first j with cum[j] >= target
 import bisect
 j = bisect.bisect_left(cum, target)
 boundaries.append(j)
chunks = []
prev = 0
for b in boundaries + [len(all_zeros_sorted)]:
 chunks.append(all_zeros_sorted[prev:b])
 prev = b
print("Chunk sizes (each balanced by T^2.68 cost):")
for i, ch in enumerate(chunks):
 Tlo = float(ch[0])
 Thi = float(ch[-1])
 print(f" chunk {i}: n={len(ch)} T=[{Tlo:.1f}, {Thi:.1f}]")

# Save chunks as pickle files
os.makedirs('/workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work/refine_inputs', exist_ok=True)
chunk_files = []
for i, ch in enumerate(chunks):
 pth = f'/workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work/refine_inputs/chunk_{i}.pkl'
 with open(pth, 'wb') as f:
 pickle.dump(ch, f)
 chunk_files.append(pth)
print("written chunks")

Chunk sizes (each balanced by T^2.68 cost):
 chunk 0: n=1770 T=[9.2, 1284.9]
 chunk 1: n=434 T=[1285.3, 1536.6]
 chunk 2: n=300 T=[1536.8, 1706.3]
 chunk 3: n=236 T=[1706.9, 1838.1]
 chunk 4: n=199 T=[1838.5, 1947.3]
 chunk 5: n=173 T=[1947.8, 2041.5]
 chunk 6: n=153 T=[2042.3, 2124.6]
 chunk 7: n=140 T=[2124.8, 2199.5]
written chunks


In [44]:
# Launch 8 parallel quick refines with deadline = 360s (6 min, given we have ~7 min budget)
import subprocess, sys, os, time
os.chdir('/workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work')
DEADLINE = 360
procs = []
os.makedirs('refine_outputs', exist_ok=True)
t_start = time.time()
for i in range(8):
 in_pkl = f'refine_inputs/chunk_{i}.pkl'
 out_pkl = f'refine_outputs/refined_{i}.pkl'
 log = open(f'refine_outputs/refine_{i}.log', 'w')
 cmd = [sys.executable, 'quick_refine.py', str(i), in_pkl, '80', out_pkl, str(DEADLINE)]
 p = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
 procs.append((p, log, i))
 print(f"launched chunk {i}: PID {p.pid}")

# Save PIDs
with open('quickrefine.pids', 'w') as f:
 for p, _, _ in procs:
 f.write(f"{p.pid}\n")
print(f"all launched at t={t_start:.0f}")

launched chunk 0: PID 197
launched chunk 1: PID 198
launched chunk 2: PID 199
launched chunk 3: PID 200
launched chunk 4: PID 201
launched chunk 5: PID 202
launched chunk 6: PID 203
launched chunk 7: PID 204
all launched at t=1780181010


In [45]:
import time
time.sleep(60)
def check():
 import os
 out_dir = '/workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work/refine_outputs'
 import pickle
 for i in range(8):
 log = f'{out_dir}/refine_{i}.log'
 try:
 with open(log) as f:
 lines = f.read().splitlines()
 print(f"chunk {i}: " + (lines[-1][:140] if lines else '(empty)'))
 except FileNotFoundError:
 print(f"chunk {i}: no log")
check()

chunk 0: [0] 425/1770 elapsed=61s
chunk 1: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
chunk 2: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
chunk 3: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
chunk 4: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
chunk 5: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
chunk 6: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
chunk 7: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072


In [46]:
# Chunk 0 made 425/1770 in 61s. Linear extrap would say ~250s for full but it slows as T grows.
# Others working silently. Let's wait some more.
import time
time.sleep(120)
check()

chunk 0: [0] 650/1770 elapsed=180s
chunk 1: [1] 25/434 elapsed=159s
chunk 2: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
chunk 3: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
chunk 4: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
chunk 5: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
chunk 6: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
chunk 7: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072


In [47]:
# chunk 0 at 650/1770 in 180s. Slowing. Chunk 1 at 25/434 in 159s (6.4s/zero average at T~1300).
# Chunk 1 estimate: 434 * 6.4 = ~2780s. Way over our 360s deadline. It will hit deadline at ~56 zeros completed.
# 
# Let me wait until close to deadline then collect partial results.
import time
time.sleep(150) # total 330s elapsed
check()

chunk 0: [0] 825/1770 elapsed=347s
chunk 1: [1] 50/434 elapsed=324s
chunk 2: [2] 25/300 elapsed=266s
chunk 3: [3] 25/236 elapsed=292s
chunk 4: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
chunk 5: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
chunk 6: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
chunk 7: PARI stack size set to 2000000000 bytes, maximum size set to 2000003072


In [48]:
# Time pressure now. Wait for deadline (only ~30 more sec), then collect everything.
import time
time.sleep(60)
check()

chunk 0: [0] DONE n=837/1770 elapsed=361s
chunk 1: [1] DONE n=56/434 elapsed=363s
chunk 2: [2] DONE n=34/300 elapsed=362s
chunk 3: [3] DONE n=31/236 elapsed=363s
chunk 4: [4] DONE n=25/199 elapsed=369s
chunk 5: [5] DONE n=21/173 elapsed=363s
chunk 6: [6] DONE n=19/153 elapsed=366s
chunk 7: [7] DONE n=17/140 elapsed=370s


In [49]:
# All 8 chunks hit deadline and dumped. Total refined: 837+56+34+31+25+21+19+17 = 1040.
# Collect, merge with dps=38 approx for unrefined zeros, save partial dps=80 list.
import pickle, os, numpy as np

os.chdir('/workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work')

refined_all = []
unrefined_T_ranges = []
for i in range(8):
 with open(f'refine_outputs/refined_{i}.pkl', 'rb') as f:
 rec = pickle.load(f)
 refined_all.extend(rec['zeros'])
 n_done = rec['n']
 # Identify which approx zeros NOT covered:
 with open(f'refine_inputs/chunk_{i}.pkl', 'rb') as f:
 chunk_input = pickle.load(f)
 if n_done < len(chunk_input):
 unrefined_T_ranges.append((i, n_done, len(chunk_input), 
 float(chunk_input[n_done]), 
 float(chunk_input[-1])))

print(f"Refined (dps=80): {len(refined_all)} zeros")
for i, nd, nt, tlo, thi in unrefined_T_ranges:
 print(f" chunk {i}: refined {nd}/{nt}; unrefined T in [{tlo:.1f}, {thi:.1f}]")

# Save just the refined dps=80 zeros sorted
refined_sorted = sorted(refined_all, key=float)
arr = np.array(refined_sorted, dtype='<U81')
out_path = f'/workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work/ldelta_zeros_N{len(arr)}_dps80.npy'
np.save(out_path, arr)
print(f"\nSaved {arr.shape[0]} dps=80 zeros to {out_path}")
print(f"first: {arr[0]}")
print(f"last: {arr[-1]}")
print(f"last T = {float(arr[-1]):.4f}")

# Validate gamma_1
GAMMA1_REF = "9.22237939992110252224376719274347813552877062243200928999818642800192129537"
nm = 0
for a_, b_ in zip(arr[0], GAMMA1_REF):
 if a_ == b_: nm += 1
 else: break
print(f"\ngamma_1 matches first {nm}/{len(GAMMA1_REF)} chars of reference")

Refined (dps=80): 1040 zeros
 chunk 0: refined 837/1770; unrefined T in [705.3, 1284.9]
 chunk 1: refined 56/434; unrefined T in [1318.1, 1536.6]
 chunk 2: refined 34/300; unrefined T in [1556.6, 1706.3]
 chunk 3: refined 31/236; unrefined T in [1724.3, 1838.1]
 chunk 4: refined 25/199; unrefined T in [1852.0, 1947.3]
 chunk 5: refined 21/173; unrefined T in [1959.1, 2041.5]
 chunk 6: refined 19/153; unrefined T in [2052.5, 2124.6]
 chunk 7: refined 17/140; unrefined T in [2134.1, 2199.5]

Saved 1040 dps=80 zeros to /workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work/ldelta_zeros_N1040_dps80.npy
first: 9.2223793999211025222437671927434781355287706224320092899981864280019212953707290
last: 2133.5478214472220848906885395965052625726690596768133865983128178553288101241818
last T = 2133.5478

gamma_1 matches first 76/76 chars of reference


In [50]:
# Note the refined zeros have a "gap" structure - chunks 1-7 refined the LOW-T zeros from each shard.
# Combined refined set covers the first ~837 zeros of chunk 0 (T<~705) + smatterings from other chunks at their start.
# The 1040 zeros are NOT contiguous in T. Let me check distribution.
import numpy as np
arr_T = np.array([float(s) for s in arr])
print(f"min T = {arr_T.min():.2f}, max T = {arr_T.max():.2f}")
print(f"distribution by T range:")
for tlo, thi in [(0,500), (500,1000), (1000,1500), (1500,2000), (2000,2200)]:
 n = ((arr_T >= tlo) & (arr_T < thi)).sum()
 print(f" [{tlo:4d}, {thi:4d}]: {n} zeros")
 
# Also save the COMPLETE search-list (dps=38 approx) as a separate artifact for posterity
import pickle
all_T_approx = [float(s) for s in all_zeros_sorted]
print(f"\nApprox (dps=38) total: {len(all_zeros_sorted)}, T_max={max(all_T_approx):.1f}")
approx_arr = np.array(all_zeros_sorted, dtype='<U51') # dps=38 -> <U51 sufficient
np.save('/workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work/ldelta_zeros_N3405_approx_dps38.npy', approx_arr)
print("saved ldelta_zeros_N3405_approx_dps38.npy")

min T = 9.22, max T = 2133.55
distribution by T range:
 [ 0, 500]: 540 zeros
 [ 500, 1000]: 297 zeros
 [1000, 1500]: 56 zeros
 [1500, 2000]: 111 zeros
 [2000, 2200]: 36 zeros

Approx (dps=38) total: 3405, T_max=2199.5
saved ldelta_zeros_N3405_approx_dps38.npy


In [51]:
# Make summary figure and finalize artifacts.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 1, figsize=(9, 8))

# Panel A: Cost-aware partition - shard wall time vs zero count
ax = axes[0]
shard_ranges = [(0, 1281.90), (1281.90, 1534.77), (1534.77, 1705.22), (1705.22, 1837.52),
 (1837.52, 1947.17), (1947.17, 2041.60), (2041.60, 2125.00), (2125.00, 2200.00)]
shard_counts = [1764, 436, 302, 237, 199, 174, 154, 139]
shard_wall = [400.5, 349.3, 314.8, 299.3, 298.2, 301.4, 300.7, 298.3]
x = np.arange(8)
w = 0.4
ax.bar(x - w/2, shard_counts, width=w, label='# zeros found (Stage 1)', color='C0')
ax2 = ax.twinx()
ax2.bar(x + w/2, shard_wall, width=w, color='C3', label='Stage-1 wall time (s)')
ax.set_xlabel('Shard index')
ax.set_ylabel('# zeros found', color='C0')
ax2.set_ylabel('Stage 1 wall time (s)', color='C3')
ax.set_title('A. Cost-aware partition (Tmax/Nshard)^(1/(1+2.85)) yields\nbalanced Stage-1 wall times (≈300–400 s) despite 13×–1764× zero-count spread',
 fontsize=10)
ax.set_xticks(x)
ax.set_xticklabels([f"s{i}\n[{tlo:.0f},{thi:.0f}]" for i, (tlo, thi) in enumerate(shard_ranges)], fontsize=8)
ax.legend(loc='upper right', fontsize=8)

# Panel B: Refined zero spacing (validate physics) + refined coverage
ax = axes[1]
T_refined = np.array(sorted(float(s) for s in arr))
spacings = np.diff(T_refined)
# Plot zero index vs T
ax.scatter(T_refined, np.arange(len(T_refined)), s=4, alpha=0.5, label=f'Refined dps=80 zeros (N={len(T_refined)})')
# overlay all approx zeros
T_approx = np.array(sorted(float(s) for s in all_zeros_sorted))
ax.plot(T_approx, np.arange(len(T_approx)), '--', alpha=0.3, color='gray',
 label=f'All dps=38 approx zeros (N={len(T_approx)})')
ax.set_xlabel('Im(zero) = T')
ax.set_ylabel('Zero index')
ax.set_title('B. Refined (dps=80) coverage vs approximate (dps=38) list.\n'
 'γ₁ matches LMFDB reference to 76/76 digits.', fontsize=10)
ax.legend(loc='lower right', fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work/ldelta_generation_summary.png', dpi=120, bbox_inches='tight')
plt.close()
print("saved figure")

saved figure


In [52]:
# Final summary
import os
work = '/workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work'
for fn in sorted(os.listdir(work)):
 fp = os.path.join(work, fn)
 if os.path.isfile(fp):
 print(f" {fn}: {os.path.getsize(fp)} bytes")

 delta_shard_runner.py: 4713 bytes
 generate_ldelta.py: 6389 bytes
 ldelta_approx_dps38_all.pkl: 143050 bytes
 ldelta_generation_summary.png: 113896 bytes
 ldelta_zeros_N1040_dps80.npy: 337088 bytes
 ldelta_zeros_N3405_approx_dps38.npy: 694748 bytes
 quick_refine.py: 2060 bytes
 quickrefine.pids: 32 bytes
 run_log.txt: 4933 bytes
 stage2.pid: 3 bytes
 stage2_log.txt: 384 bytes
 stage2_only.py: 2439 bytes


In [53]:
from e14c.filesystem import edit_file
mem_path = '/workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/.prompts/MEMORY.md'
with open(mem_path) as f:
 txt = f.read()
new_txt = txt.replace('[Fill me in]', '2026-05-30: L(Delta,s) hybrid search/refine implementation tested in /workspace/3cd100c8-68f7-45e1-af7a-0810da5187e0/work.\n- Modified working copies: work/generate_ldelta.py, work/delta_shard_runner.py\n- Stage-1 cost-aware partition with exponent 2.85 validated at T_max=2200: shard wall times 298.2-400.5 s despite zero-count imbalance 139-1764; total approx zeros found = 3405.\n- Stage-2 secant refinement using PARI lfunhardy validated against high-precision lfunzeros on test zero near T~1500: 2 secant iterations reproduced 80-digit zero exactly (81 matching chars including decimal).\n- Full N=5000,dps=80 generation not completed within 3600 s runtime budget. Partial outputs created: work/ldelta_zeros_N1040_dps80.npy and work/ldelta_zeros_N3405_approx_dps38.npy.\n- gamma_1 from refined output matched 76/76 chars of reference.\n- Provenance: current task, files in work/, run logs in work/run_log.txt and work/refine_outputs/*.log.\n\n[Fill me in]', 1)
with open(mem_path, 'w') as f:
 f.write(new_txt)
print('updated MEMORY.md')

updated MEMORY.md
